Notebook purpose (how this analysis supports the thesis)
This notebook operationalises the thesis proposal “Governance‑Ready Fraud Decisioning for Transaction Monitoring: Evidence‑Grounded Narrative Explanations with Tiered Guardrails” (Andrew Murtagh, 22 Feb 2026) by analysing a minimal, reproducible run of the pipeline:

a leakage‑controlled baseline fraud model on IEEE‑CIS (temporal quantile split),
per‑event Evidence Objects (EOs) as the closed‑world “contract” for explanations, and
persona‑specific narratives generated under two explanation conditions (templates vs constrained LLM), validated by a tiered validator with retries and template fallback.
The outputs here are intended for thesis figures/tables and for diagnosing governance‑relevant failure modes (schema violations, closed‑world breaches, missing disclosures, and action inconsistency).

What is being tested (research questions mapped to notebook outputs)
This analysis focuses on the proposal’s core evaluation axes:

Faithfulness (RQ1): Do narratives preserve EO driver membership/order/direction and remain consistent with the EO’s recommended action class?
Stability (RQ2): Where available, do narratives behave predictably under perturbations or controlled stress (e.g., thin‑file / low‑coverage conditions)?
Note: The notebook now includes a compact perturbation experiment, but full perturbation/randomisation sanity checks remain outside the scope of this run.
Decision utility (RQ3, proxy): Do persona narratives provide a usable summary (risk, drivers, disclosures, action) without introducing unsupported facts?
Governance contract: Evidence Objects (EOs) and closed‑world grounding
The EO is treated as the single source of truth for downstream narratives. Narratives must remain “closed‑world” grounded: they may reference only fields present in the EO (score/band/action, top‑K drivers and directions, monitoring/drift flags, and evidence‑strength/coverage signals). Any narrative content that cannot be traced to EO fields is considered a governance failure.

Compared narrative conditions
This notebook compares (or prepares comparison for) two explanation conditions defined in the proposal:

Template narratives (deterministic baseline): guaranteed schema/grounding by construction.
Constrained LLM narratives (structured outputs): LLM returns JSON constrained to a strict schema; outputs are accepted only if they pass a defence‑in‑depth validator. Failures are retried; persistent failures trigger template fallback. Retry counts and fallback reasons are treated as empirical results, not just engineering details.
Thin‑file and monitoring disclosures (required governance behaviour)
A central requirement is correct uncertainty communication when evidence is limited. For EOs flagged as thin‑file / low evidence_strength / low coverage, narratives must include an evidence limitation disclosure (and, where applicable, drift monitoring disclosures). The notebook explicitly tracks whether disclosures appear when required, and how often validation fails due to missing disclosures.

Reproducibility and scope of this notebook
This notebook is run against the following reproducible artifacts (EO JSONL, narrative JSONL, split manifest, and metrics files) produced by the pipeline scripts. The intent is not to claim production‑grade fraud performance, but to evaluate whether the EO → narrative → validator protocol can produce governance‑ready explanations with measurable faithfulness and robust handling of thin‑file cases.

# Analysis roadmap

This notebook evaluates a reproducible run of the EO → narrative → validation pipeline for governance-ready fraud explanation analysis. The notebook is organised into the following sections.

## 1. Setup and reproducibility
**Cell 0 — Configuration and data loading**

Loads reproducible artifacts, resolves repository-relative paths, applies fail-fast checks for required files, and constructs the core analysis tables used throughout the notebook.

## 2. Input artifact inspection
**Cell 1 — EO quick inspection**  
**Cell 2 — LLM narrative quick inspection**  
**Cell 3 — EO × LLM merged inspection**

Performs lightweight sanity checks on the loaded Evidence Objects (EOs), generated LLM narratives, and the merged EO–LLM analysis set. These cells confirm that the expected fields are present and that event identifiers align correctly.

## 3. Canonical analysis frames
**Cell 4 — Canonical analysis frames + guardrails**

Constructs canonical analysis tables for LLM outputs, template outputs, and cross-condition comparison. Narrative fields are flattened into analysis-friendly columns while preserving the original structured records.

## 4. Faithfulness evaluation
**Cell 5 — Faithfulness metrics**

Computes first-order faithfulness indicators from EO-grounded narratives:
- driver overlap with EO top-K drivers,
- direction consistency between EO drivers and narrative statements,
- action consistency between EO recommendation and narrative recommendation.

These metrics operationalise the notebook’s main RQ1 checks.

## 5. Summary tables for thesis reporting
**Cell 6 — compact summary table for thesis text/tables**

Produces a compact headline summary of the main LLM-condition results for direct reuse in thesis tables or results text.

## 6. Stratified governance analysis
**Cell 7 — metrics by evidence strength**  
**Cell 8 — metrics by thin-file flag**  
**Cell 9 — check whether required evidence disclosures appear for thin-file cases**

Examines whether explanation behaviour differs under lower-evidence conditions, especially for thin-file cases. These cells focus on governance-relevant uncertainty communication and disclosure compliance.

## 7. Embedded-vs-recomputed metric checks
**Cell 10 — compare notebook-computed metrics with embedded metrics from LLM narrative JSONL**

Compares notebook-recomputed metrics against metrics embedded in the narrative artifacts. This serves as a reproducibility and consistency check on pipeline outputs.

## 8. Cross-condition normalisation for comparison
**Cell 11 — normalise LLM and template outputs into a common comparison schema**

Normalises LLM and template outputs into a single common schema to support condition comparison, downstream aggregation, and comparative tables/figures.

## Interpretation note
This notebook is intended to assess governance-readiness of explanation outputs rather than to claim production-grade fraud detection performance. Reported results should therefore be interpreted as evidence about the EO-grounded explanation protocol, disclosure behaviour, and validation robustness under the current reproducible run.

In [336]:
# Cell 0 — Configuration and data loading

from __future__ import annotations

import json
import os
from pathlib import Path
from typing import Any, Dict, List, Optional

import pandas as pd

# Optional plotting dependency
try:
    import matplotlib.pyplot as plt  # noqa: F401
    HAS_MPL = True
except Exception:
    plt = None  # type: ignore
    HAS_MPL = False


def _find_repo_root(start: Optional[Path] = None) -> Path:
    """Walk upward until a repository marker is found."""
    cur = (start or Path.cwd()).resolve()
    for p in [cur, *cur.parents]:
        if (p / "pyproject.toml").exists() or (p / ".git").exists():
            return p
    return cur


def read_jsonl(path: Path, *, limit: int | None = None) -> List[Dict[str, Any]]:
    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path}")
    rows: List[Dict[str, Any]] = []
    with path.open("r", encoding="utf-8") as f:
        for i, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError as e:
                raise ValueError(f"Bad JSON at {path} line {i}: {e}") from e
            if limit is not None and len(rows) >= limit:
                break
    return rows


def _assert_columns(df: pd.DataFrame, cols: List[str], *, name: str) -> None:
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise AssertionError(
            f"{name} missing required columns: {missing}\n"
            f"Found: {list(df.columns)}"
        )


REPO_ROOT = _find_repo_root()
ARTIFACTS = REPO_ROOT / "artifacts" / "baselines" / "lgbm_numeric_v1_subsample"
OUTPUT = REPO_ROOT / "output"

ARTIFACTS = Path(os.getenv("THESIS_ARTIFACTS_DIR", str(ARTIFACTS))).expanduser().resolve()
OUTPUT = Path(os.getenv("THESIS_OUTPUT_DIR", str(OUTPUT))).expanduser().resolve()

EO_PATH = ARTIFACTS / "eos_test_with_drivers.jsonl"
OPS_TRIAGE_LLM_PATH = OUTPUT / "narratives_ops_triage_llm.jsonl"
OPS_TRIAGE_TEMPLATE_PATH = OUTPUT / "narratives_ops_triage_template.jsonl"

METRICS_PATH = ARTIFACTS / "metrics.json"
THIN_THICK_PATH = ARTIFACTS / "thin_thick_metrics.json"

required = {
    "EO_PATH": EO_PATH,
    "OPS_TRIAGE_LLM_PATH": OPS_TRIAGE_LLM_PATH,
    "OPS_TRIAGE_TEMPLATE_PATH": OPS_TRIAGE_TEMPLATE_PATH,
}
missing = {k: str(p) for k, p in required.items() if not p.exists()}
if missing:
    msg = "\n".join([f"- {k}: {v}" for k, v in missing.items()])
    raise FileNotFoundError(
        "Required artifact(s) not found:\n"
        f"{msg}\n\n"
        "Set THESIS_ARTIFACTS_DIR / THESIS_OUTPUT_DIR or regenerate the pipeline outputs."
    )

eo_rows = read_jsonl(EO_PATH)
eo_df = pd.DataFrame(eo_rows)
_assert_columns(
    eo_df,
    ["event_id", "top_drivers", "recommended_action_class", "thin_file_flag", "evidence_strength"],
    name="eo_df",
)
eo_df["event_id"] = eo_df["event_id"].astype(str)

llm_rows = read_jsonl(OPS_TRIAGE_LLM_PATH)
llm_df = pd.DataFrame(llm_rows)
_assert_columns(llm_df, ["eo_event_id", "persona", "engine", "narrative", "metrics"], name="llm_df")
llm_df["eo_event_id"] = llm_df["eo_event_id"].astype(str)

tmpl_rows = read_jsonl(OPS_TRIAGE_TEMPLATE_PATH)
tmpl_df = pd.DataFrame(tmpl_rows) if tmpl_rows else pd.DataFrame()

eo_llm_df = eo_df.merge(
    llm_df,
    left_on="event_id",
    right_on="eo_event_id",
    how="inner",
    suffixes=("_eo", "_llm"),
)
if eo_llm_df.empty:
    raise AssertionError(
        "Merged EO × LLM is empty (likely event_id mismatch).\n"
        f"Example EO event_id: {eo_df['event_id'].iloc[0] if len(eo_df) else 'N/A'}\n"
        f"Example LLM eo_event_id: {llm_df['eo_event_id'].iloc[0] if len(llm_df) else 'N/A'}"
    )

print("REPO_ROOT:", REPO_ROOT)
print("ARTIFACTS:", ARTIFACTS)
print("OUTPUT:", OUTPUT)
print("HAS_MPL:", HAS_MPL, "(matplotlib optional)")
print(f"EO rows: {len(eo_df)} | LLM rows: {len(llm_df)} | eo_llm rows: {len(eo_llm_df)}")
print("EO thin_file_flag rate:", float(eo_df["thin_file_flag"].mean()))
print(
    "Optional metrics present:",
    "metrics.json" if METRICS_PATH.exists() else "metrics.json (missing)",
    "|",
    "thin_thick_metrics.json" if THIN_THICK_PATH.exists() else "thin_thick_metrics.json (missing)",
)

assert len(eo_df) > 0
assert len(llm_df) > 0
assert len(eo_llm_df) > 0

REPO_ROOT: /workspaces/miniature-fishstick
ARTIFACTS: /workspaces/miniature-fishstick/artifacts/baselines/lgbm_numeric_v1_subsample
OUTPUT: /workspaces/miniature-fishstick/output
HAS_MPL: True (matplotlib optional)
EO rows: 20000 | LLM rows: 200 | eo_llm rows: 200
EO thin_file_flag rate: 0.7909
Optional metrics present: metrics.json | thin_thick_metrics.json


In [337]:
# Cell 1 — EO quick inspection
print("Number of EO records:", len(eo_df))
eo_df[[
    "event_id",
    "score",
    "risk_band",
    "recommended_action_class",
    "thin_file_flag",
    "evidence_strength"
]].head(5)

Number of EO records: 20000


,event_id,score,risk_band,recommended_action_class,thin_file_flag,evidence_strength
0,3488961,0.005848,LOW,allow,True,LOW
1,3488968,0.000505,LOW,allow,True,LOW
2,3488985,0.004906,LOW,allow,True,LOW
3,3489002,0.002055,LOW,allow,True,LOW
4,3489003,0.016481,LOW,allow,True,LOW


In [338]:
# Cell 2 — LLM narrative quick inspection
print("Number of LLM narrative records:", len(llm_df))
llm_df[["eo_event_id", "persona", "engine"]].head(5)

Number of LLM narrative records: 200


,eo_event_id,persona,engine
0,3488961,ops_triage,llm
1,3488968,ops_triage,llm
2,3488985,ops_triage,llm
3,3489002,ops_triage,llm
4,3489003,ops_triage,llm


In [339]:
# Cell 3 — EO × LLM merged inspection

print("Number of EO × LLM merged records:", len(eo_llm_df))
eo_llm_df[
    [
        "event_id",
        "score",
        "risk_band",
        "recommended_action_class",
        "thin_file_flag",
        "evidence_strength",
    ]
].head(5)

Number of EO × LLM merged records: 200


,event_id,score,risk_band,recommended_action_class,thin_file_flag,evidence_strength
0,3488961,0.005848,LOW,allow,True,LOW
1,3488968,0.000505,LOW,allow,True,LOW
2,3488985,0.004906,LOW,allow,True,LOW
3,3489002,0.002055,LOW,allow,True,LOW
4,3489003,0.016481,LOW,allow,True,LOW


In [340]:
# Cell 4 — Canonical analysis frames + guardrails


# --- helper: flatten selected narrative fields safely ---
def _extract_narrative_fields(df: pd.DataFrame, narrative_col: str = "narrative") -> pd.DataFrame:
    out = df.copy()

    def _get(d, key, default=None):
        return d.get(key, default) if isinstance(d, dict) else default

    out["narrative_summary"] = out[narrative_col].apply(lambda d: _get(d, "summary"))
    out["narrative_action_recommendation"] = out[narrative_col].apply(lambda d: _get(d, "action_recommendation"))
    out["narrative_drivers_used"] = out[narrative_col].apply(lambda d: _get(d, "drivers_used", []))
    out["narrative_driver_statements"] = out[narrative_col].apply(lambda d: _get(d, "driver_statements", []))
    out["narrative_disclosures"] = out[narrative_col].apply(lambda d: _get(d, "disclosures", []))
    out["narrative_monitoring_flags"] = out[narrative_col].apply(lambda d: _get(d, "monitoring_flags", []))
    return out

# --- canonical LLM frame ---
llm_analysis_df = _extract_narrative_fields(eo_llm_df)

# --- canonical template frame ---
if len(tmpl_df) > 0:
    tmpl_df = tmpl_df.copy()
    if "eo_event_id" in tmpl_df.columns:
        tmpl_df["eo_event_id"] = tmpl_df["eo_event_id"].astype(str)

    tmpl_eo_df = eo_df.merge(
        tmpl_df,
        left_on="event_id",
        right_on="eo_event_id",
        how="inner",
        suffixes=("_eo", "_tmpl"),
    )
    tmpl_analysis_df = _extract_narrative_fields(tmpl_eo_df)
else:
    tmpl_eo_df = pd.DataFrame()
    tmpl_analysis_df = pd.DataFrame()

# --- comparison frame: LLM joined to template on event_id ---
if not tmpl_analysis_df.empty:
    llm_cols = [
        "event_id",
        "score",
        "risk_band",
        "recommended_action_class",
        "thin_file_flag",
        "evidence_strength",
        "top_drivers",
        "narrative",
        "metrics",
        "narrative_summary",
        "narrative_action_recommendation",
        "narrative_drivers_used",
        "narrative_driver_statements",
        "narrative_disclosures",
        "narrative_monitoring_flags",
        "overlap_at_5",
        "direction_accuracy",
        "action_consistency",
    ]
    llm_cols = [c for c in llm_cols if c in llm_analysis_df.columns]

    tmpl_cols = [
        "event_id",
        "narrative",
        "metrics",
        "narrative_summary",
        "narrative_action_recommendation",
        "narrative_drivers_used",
        "narrative_driver_statements",
        "narrative_disclosures",
        "narrative_monitoring_flags",
    ]
    tmpl_cols = [c for c in tmpl_cols if c in tmpl_analysis_df.columns]

    compare_df = llm_analysis_df[llm_cols].merge(
        tmpl_analysis_df[tmpl_cols],
        on="event_id",
        how="inner",
        suffixes=("_llm", "_tmpl"),
    )
else:
    compare_df = pd.DataFrame()

print("llm_analysis_df:", llm_analysis_df.shape)
print("tmpl_analysis_df:", tmpl_analysis_df.shape)
print("compare_df:", compare_df.shape)

llm_analysis_df: (200, 28)
tmpl_analysis_df: (200, 28)
compare_df: (200, 23)


In [341]:
# Cell 5 — Faithfulness metrics

def overlap_at_k(row, k=5):
    top_drivers = row.get("top_drivers", [])
    narrative = row.get("narrative", {})
    if not isinstance(top_drivers, list) or not isinstance(narrative, dict):
        return None
    eo_drivers = [d.get("name") for d in top_drivers[:k] if isinstance(d, dict)]
    llm_drivers = narrative.get("drivers_used", [])
    if not eo_drivers or not isinstance(llm_drivers, list):
        return None
    return len(set(eo_drivers) & set(llm_drivers)) / float(k)


def direction_accuracy(row, k=5):
    top_drivers = row.get("top_drivers", [])
    narrative = row.get("narrative", {})
    if not isinstance(top_drivers, list) or not isinstance(narrative, dict):
        return None
    eo_drv = {
        d.get("name"): d.get("direction")
        for d in top_drivers[:k]
        if isinstance(d, dict) and d.get("name") is not None
    }
    llm_statements = narrative.get("driver_statements", [])
    if not isinstance(llm_statements, list):
        return None
    llm_dir = {
        d.get("name"): d.get("direction")
        for d in llm_statements
        if isinstance(d, dict) and d.get("name") is not None
    }
    overlap_names = set(eo_drv.keys()) & set(llm_dir.keys())
    if not overlap_names:
        return None
    correct = sum(eo_drv[name] == llm_dir[name] for name in overlap_names)
    return correct / len(overlap_names)


def action_consistency(row):
    narrative = row.get("narrative", {})
    if not isinstance(narrative, dict):
        return None
    eo_action = row.get("recommended_action_class")
    llm_action = narrative.get("action_recommendation")
    return eo_action == llm_action


eo_llm_df["overlap_at_5"] = eo_llm_df.apply(overlap_at_k, axis=1)
eo_llm_df["direction_accuracy"] = eo_llm_df.apply(direction_accuracy, axis=1)
eo_llm_df["action_consistency"] = eo_llm_df.apply(action_consistency, axis=1)

print("Mean overlap@5:", eo_llm_df["overlap_at_5"].mean())
print("Mean direction accuracy:", eo_llm_df["direction_accuracy"].mean())
print("Action consistency rate:", eo_llm_df["action_consistency"].mean())

Mean overlap@5: 1.0
Mean direction accuracy: 1.0
Action consistency rate: 1.0


In [342]:
# Cell 6 — compact summary table for thesis text/tables

llm_summary_df = pd.DataFrame({
    "metric": [
        "n_eo_llm",
        "mean_overlap_at_5",
        "mean_direction_accuracy",
        "action_consistency_rate",
        "thin_file_rate"
    ],
    "value": [
        len(eo_llm_df),
        eo_llm_df["overlap_at_5"].mean(),
        eo_llm_df["direction_accuracy"].mean(),
        eo_llm_df["action_consistency"].mean(),
        eo_llm_df["thin_file_flag"].mean()
    ]
})

llm_summary_df

,metric,value
0,n_eo_llm,200.0
1,mean_overlap_at_5,1.0
2,mean_direction_accuracy,1.0
3,action_consistency_rate,1.0
4,thin_file_rate,1.0


In [343]:
# Cell 7 — metrics by evidence strength

llm_by_evidence_df = eo_llm_df.groupby("evidence_strength").agg(
    n=("event_id", "count"),
    overlap_at_5_mean=("overlap_at_5", "mean"),
    direction_accuracy_mean=("direction_accuracy", "mean"),
    action_consistency_rate=("action_consistency", "mean"),
)

llm_by_evidence_df

,n,overlap_at_5_mean,direction_accuracy_mean,action_consistency_rate
evidence_strength,,,,
LOW,200,1.0,1.0,1.0


In [344]:
# Cell 8 — metrics by thin-file flag

llm_by_thin_file_df = eo_llm_df.groupby("thin_file_flag").agg(
    n=("event_id", "count"),
    overlap_at_5_mean=("overlap_at_5", "mean"),
    direction_accuracy_mean=("direction_accuracy", "mean"),
    action_consistency_rate=("action_consistency", "mean"),
)

llm_by_thin_file_df

,n,overlap_at_5_mean,direction_accuracy_mean,action_consistency_rate
thin_file_flag,,,,
True,200,1.0,1.0,1.0


In [345]:
# Cell 9 — check whether required evidence disclosures appear for thin-file cases

def has_evidence_disclosure(narr):
    if not isinstance(narr, dict):
        return False
    disclosures = narr.get("disclosures", {})
    if not isinstance(disclosures, dict):
        return False
    val = disclosures.get("evidence")
    return isinstance(val, str) and len(val.strip()) > 0

eo_llm_df["has_evidence_disclosure"] = eo_llm_df["narrative"].apply(has_evidence_disclosure)

llm_disclosure_summary_df = eo_llm_df.groupby("thin_file_flag").agg(
    n=("event_id", "count"),
    disclosure_rate=("has_evidence_disclosure", "mean"),
)

llm_disclosure_summary_df

,n,disclosure_rate
thin_file_flag,,
True,200,1.0


In [346]:
# Cell 10 — compare notebook-computed metrics with embedded metrics from LLM narrative JSONL

print("Template rows:", len(tmpl_df))
print("Template columns:", list(tmpl_df.columns))

def _metric_from_metrics_col(df: pd.DataFrame, key: str):
    if "metrics" not in df.columns:
        return pd.Series(dtype=float)
    vals = df["metrics"].apply(
        lambda d: d.get(key) if isinstance(d, dict) else None
    )
    return pd.to_numeric(vals, errors="coerce")

llm_embedded_overlap = _metric_from_metrics_col(llm_df, "overlap_at_K")
llm_embedded_direction = _metric_from_metrics_col(llm_df, "direction_accuracy")
llm_embedded_action = _metric_from_metrics_col(llm_df, "action_consistency")

metric_compare_df = pd.DataFrame([{
    "notebook_overlap_mean": eo_llm_df["overlap_at_5"].mean(),
    "embedded_overlap_mean": llm_embedded_overlap.mean(),
    "notebook_direction_mean": eo_llm_df["direction_accuracy"].mean(),
    "embedded_direction_mean": llm_embedded_direction.mean(),
    "notebook_action_mean": eo_llm_df["action_consistency"].mean(),
    "embedded_action_mean": llm_embedded_action.mean(),
}])

metric_compare_df

Template rows: 200
Template columns: ['eo_event_id', 'persona', 'engine', 'run_metadata', 'narrative', 'metrics']


,notebook_overlap_mean,embedded_overlap_mean,notebook_direction_mean,embedded_direction_mean,notebook_action_mean,embedded_action_mean
0,1.0,1.0,1.0,1.0,1.0,1.0


In [347]:
# Cell 11 — normalise LLM and template outputs into a common comparison schema

def extract_llm_fields(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    out["event_id_norm"] = out["eo_event_id"].astype(str)
    out["condition"] = "llm"

    out["summary_text"] = out["narrative"].apply(
        lambda x: x.get("summary") if isinstance(x, dict) else None
    )
    out["drivers_used_norm"] = out["narrative"].apply(
        lambda x: x.get("drivers_used", []) if isinstance(x, dict) else []
    )
    out["driver_statements_norm"] = out["narrative"].apply(
        lambda x: x.get("driver_statements", []) if isinstance(x, dict) else []
    )
    out["action_recommendation_norm"] = out["narrative"].apply(
        lambda x: x.get("action_recommendation") if isinstance(x, dict) else None
    )
    out["disclosures_norm"] = out["narrative"].apply(
        lambda x: x.get("disclosures", {}) if isinstance(x, dict) else {}
    )

    keep = [
        "event_id_norm",
        "condition",
        "persona",
        "engine",
        "summary_text",
        "drivers_used_norm",
        "driver_statements_norm",
        "action_recommendation_norm",
        "disclosures_norm",
        "metrics",
    ]
    keep = [c for c in keep if c in out.columns]
    return out[keep]


def extract_template_fields(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    if "event_id" in out.columns:
        out["event_id_norm"] = out["event_id"].astype(str)
    elif "eo_event_id" in out.columns:
        out["event_id_norm"] = out["eo_event_id"].astype(str)
    else:
        raise AssertionError("Template dataframe needs either 'event_id' or 'eo_event_id'")

    out["condition"] = "template"
    if "persona" not in out.columns:
        out["persona"] = None
    if "engine" not in out.columns:
        out["engine"] = "template"

    out["summary_text"] = out["narrative"].apply(
        lambda x: x.get("summary") if isinstance(x, dict) else None
    )
    out["drivers_used_norm"] = out["narrative"].apply(
        lambda x: x.get("drivers_used", []) if isinstance(x, dict) else []
    )
    out["driver_statements_norm"] = out["narrative"].apply(
        lambda x: x.get("driver_statements", []) if isinstance(x, dict) else []
    )
    out["action_recommendation_norm"] = out["narrative"].apply(
        lambda x: x.get("action_recommendation") if isinstance(x, dict) else None
    )
    out["disclosures_norm"] = out["narrative"].apply(
        lambda x: x.get("disclosures", {}) if isinstance(x, dict) else {}
    )

    if "metrics" not in out.columns:
        out["metrics"] = None

    keep = [
        "event_id_norm",
        "condition",
        "persona",
        "engine",
        "summary_text",
        "drivers_used_norm",
        "driver_statements_norm",
        "action_recommendation_norm",
        "disclosures_norm",
        "metrics",
    ]
    keep = [c for c in keep if c in out.columns]
    return out[keep]


llm_norm_df = extract_llm_fields(llm_df)
tmpl_norm_df = extract_template_fields(tmpl_df) if len(tmpl_df) > 0 else pd.DataFrame()

combined_norm_df = pd.concat(
    [df for df in [llm_norm_df, tmpl_norm_df] if not df.empty],
    ignore_index=True,
)

print("LLM normalised rows:", len(llm_norm_df))
print("Template normalised rows:", len(tmpl_norm_df))
print("Combined rows:", len(combined_norm_df))

combined_norm_df.head()

LLM normalised rows: 200
Template normalised rows: 200
Combined rows: 400


,event_id_norm,condition,persona,engine,summary_text,drivers_used_norm,driver_statements_norm,action_recommendation_norm,disclosures_norm,metrics
0,3488961,llm,ops_triage,llm,Risk is LOW (score 0.005848). TransactionAmt i...,"[TransactionAmt, C5, C13, D1, D10]","[{'name': 'TransactionAmt', 'direction': '+', ...",allow,{'evidence': 'Thin-file / low coverage: limite...,"{'overlap_at_K': 1.0, 'direction_accuracy': Tr..."
1,3488968,llm,ops_triage,llm,Risk is LOW (score 0.000505). C5 reduces risk....,"[C5, card5, C9, C14, D1]","[{'name': 'C5', 'direction': '-', 'text': 'C5 ...",allow,{'evidence': 'Thin-file / low coverage: limite...,"{'overlap_at_K': 1.0, 'direction_accuracy': Tr..."
2,3488985,llm,ops_triage,llm,Risk is LOW (score 0.004906). C14 reduces risk...,"[C14, C5, C1, C9, card2]","[{'name': 'C14', 'direction': '-', 'text': 'C1...",allow,{'evidence': 'Thin-file / low coverage: limite...,"{'overlap_at_K': 1.0, 'direction_accuracy': Tr..."
3,3489002,llm,ops_triage,llm,Risk is LOW (score 0.002055). D2 reduces risk....,"[D2, TransactionAmt, D1, C5, card2]","[{'name': 'D2', 'direction': '-', 'text': 'D2 ...",allow,{'evidence': 'Thin-file / low coverage: limite...,"{'overlap_at_K': 1.0, 'direction_accuracy': Tr..."
4,3489003,llm,ops_triage,llm,Risk is LOW (score 0.016481). card5 increases ...,"[card5, C5, D10, C13, TransactionDT]","[{'name': 'card5', 'direction': '+', 'text': '...",allow,{'evidence': 'Thin-file / low coverage: limite...,"{'overlap_at_K': 1.0, 'direction_accuracy': Tr..."


## Results: EO-grounded narrative governance evaluation

This section evaluates a reproducible run of the proposed EO → narrative → validator pipeline in the IEEE-CIS fraud setting. The analysis focuses on whether downstream narratives remain faithful to Evidence Objects (EOs), communicate uncertainty appropriately under limited evidence, and satisfy the closed-world governance constraints defined in the thesis.

The underlying fraud detector provides a credible substrate for explanation analysis rather than the primary object of optimisation. On the temporal split v1_temporal_q70_q85, the leakage-controlled LightGBM baseline achieved validation ROC-AUC of 0.8722 and PR-AUC of 0.4478, with test ROC-AUC of 0.8687 and PR-AUC of 0.4594. These results indicate that the explanation study is grounded in a non-trivial and reasonably competent fraud-scoring model, without claiming state-of-the-art predictive performance.

The full EO artifact contains 20,000 test-side records. The detailed narrative evaluation reported here is conducted on a 200-event sampled slice for the constrained-LLM condition and 200 corresponding template narratives for shared-event comparison. Within this sampled run, the constrained LLM narratives achieved perfect scores on the implemented first-order faithfulness checks: overlap@5 = 1.0, direction accuracy = 1.0, and action consistency = 1.0. Required evidence disclosures for thin-file cases were also present in all sampled narratives, yielding a thin-file disclosure rate of 1.0. Notebook-recomputed metrics matched the embedded metrics stored in the narrative artifacts, supporting the internal consistency of the evaluation pipeline.

These findings indicate that, under the current EO schema, constrained generation setup, and validator configuration, the EO-grounded narrative protocol behaves as intended on the sampled slice. The generated narratives preserved EO driver membership and sign, matched EO-recommended actions under the implemented evaluation logic, and included required evidence-limit disclosures under sparse-evidence conditions. In practical terms, this suggests that a tightly constrained explanation layer can deliver structurally faithful and governance-compliant narratives without introducing unsupported content.

The broader artifact-level evaluation strengthens this conclusion. Across the three sampled personas examined—ops triage, governance audit, and engineering debug—all constrained LLM outputs passed validation on the first attempt, with no retries and no recorded failure reasons. In the ops-triage top-5 grounding analysis, no unsupported driver leakage was detected, overlap with EO top drivers was perfect, and no sign errors were observed under the implemented checks. These results provide evidence that the closed-world grounding contract was preserved in the evaluated run and that the validator architecture was effective in enforcing the required narrative schema.

A compact perturbation experiment provides more direct evidence on narrative stability. A sampled subset of 100 EO rows was expanded into 300 perturbed variants covering three controlled perturbation types: action flips, thin-file disclosure toggles, and top-driver sign flips. The resulting narratives responded correctly to thin-file disclosure toggles in all evaluated cases and to top-driver sign flips in all evaluated cases where the target driver was mentioned, indicating that the explanation layer is responsive to relevant changes in the EO contract rather than merely replaying a static template. This strengthens the thesis support for stability by showing that the narrative output changes when governance-relevant EO fields are changed.

The action-perturbation results were more nuanced. Direct response to action flips was substantially weaker than response to disclosure and sign perturbations. However, qualitative inspection shows that this is largely explained by the thin-file guardrail policy encoded in the orchestration logic: when the perturbed EO still satisfied the thin-file or low-evidence condition, the narrative system frequently overrode the EO action field and emitted a step-up recommendation instead. This is an important governance result in its own right. It shows that explanation behaviour in this system is mediated not only by direct EO field values, but also by higher-priority policy rules applied to those values. Accordingly, lower action-flip response should not be interpreted simply as instability; rather, it reflects policy-conditioned responsiveness.

Cross-condition comparison against the deterministic template baseline showed exact agreement on the sampled shared events. On the 200 shared EO inputs, constrained LLM and template narratives matched exactly in action recommendation, driver set, and evidence-disclosure presence. This does not establish superiority of one condition over the other in readability or analyst utility, but it does show that the constrained LLM condition remained structurally aligned with the deterministic baseline while preserving the same EO-grounded content.

A related governance distinction emerges in the thin-file analysis. Narrative-level action consistency and disclosure compliance were both perfect in the sampled thin-file cases, yet the stricter thin-file action-validity metric remained at 0.0 under the policy rule encoded in the evaluation artifacts. This distinction is analytically important. It shows that explanation-level faithfulness and policy-level action validity are not the same thing: a narrative can be perfectly faithful to the EO and still surface an underlying governance issue in the EO’s recommended action under limited-evidence conditions. The present results therefore support strong claims about structural faithfulness and disclosure compliance, but only partial claims about the adequacy of thin-file action policy.

Taken together, the results support the thesis claim that governance-ready explanatory narratives can be produced when explanation generation is tightly coupled to an EO contract and protected by tiered validation. Within the tested configuration, the proposed pipeline achieved strong structural faithfulness, maintained closed-world grounding, and reliably communicated evidence limitations. The added perturbation analysis strengthens this claim by showing that the narrative layer is conditionally responsive to controlled EO changes, while also revealing that such responsiveness is shaped by explicit governance policy. The remaining limitations therefore concern not whether the protocol can enforce grounded explanation behaviour in a controlled setting, but how far those guarantees extend under broader perturbation regimes, richer evidence variation, and alternative policy configurations.

## Limitations

This notebook reports first-order governance evidence together with a compact perturbation experiment, rather than the full perturbation, randomisation, or counterfactual stress-testing programme proposed for the thesis. Evidence-strength variation within the attached EO–LLM sample remains limited, and the sampled LLM subset is still heavily concentrated in thin-file cases. In addition, embedded artifact metric names and notebook-recomputed metric names are not yet fully harmonised (for example, `overlap_at_K` versus `overlap_at_5`).

More broadly, the notebook evaluates explanation-governance behaviour under a reproducible offline run; it does not, by itself, establish production deployment readiness, portfolio-level calibration adequacy, or generalisation beyond the current experiment configuration.

The uniformly perfect validator and grounding metrics in this sampled run should be interpreted cautiously: they likely reflect a strongly constrained generation setting and a relatively narrow evaluation slice, rather than demonstrating unconstrained narrative robustness.

The perturbation experiment strengthens the stability analysis, but it remains compact in scope. It covers three controlled perturbation families on a 100-event sampled subset and therefore should be interpreted as evidence of conditional responsiveness under the implemented governance logic, not as an exhaustive robustness study.

In [348]:
# Cell 12 — build perturbed EO variants for a minimal stability experiment

from copy import deepcopy
import json
import random
from pathlib import Path
import pandas as pd

# ----------------------------
# Configuration
# ----------------------------
PERTURB_N = 100          # increase to 200 if you want
PERTURB_SEED = 42
WRITE_JSONL = True

# Optional: prefer rows with at least one top driver having an explicit +/- direction
REQUIRE_TOP1_DIRECTION = True

random.seed(PERTURB_SEED)

# ----------------------------
# Helpers
# ----------------------------
def _safe_top_drivers(x):
    return x if isinstance(x, list) else []

def _flip_action(action: str) -> str:
    # Conservative action perturbation map.
    # Adjust if your EO action space differs.
    action_map = {
        "allow": "review",
        "review": "block",
        "block": "review",
        "monitor": "review",
        "decline": "review",
    }
    return action_map.get(action, "review")

def _flip_direction(direction):
    if direction == "+":
        return "-"
    if direction == "-":
        return "+"
    return direction

def _row_has_top1_direction(row) -> bool:
    top_drivers = _safe_top_drivers(row.get("top_drivers"))
    if not top_drivers:
        return False
    top1 = top_drivers[0]
    return isinstance(top1, dict) and top1.get("direction") in {"+", "-"}

def _make_variant(base_row: dict, perturbation_type: str) -> dict:
    row = deepcopy(base_row)

    # Keep original ID around; create a variant-specific event_id for generation
    base_event_id = str(row["event_id"])
    row["source_event_id"] = base_event_id
    row["perturbation_type"] = perturbation_type
    row["is_perturbed"] = True

    if perturbation_type == "action_flip":
        row["recommended_action_class_original"] = row.get("recommended_action_class")
        row["recommended_action_class"] = _flip_action(row.get("recommended_action_class"))

    elif perturbation_type == "thin_file_toggle":
        row["thin_file_flag_original"] = row.get("thin_file_flag")
        row["thin_file_flag"] = not bool(row.get("thin_file_flag"))

        # Optional: nudge evidence_strength to remain semantically aligned
        # Only do this if your generator uses evidence_strength directly.
        if "evidence_strength" in row:
            if row["thin_file_flag"] is True:
                row["evidence_strength"] = "LOW"
            elif row["evidence_strength"] == "LOW":
                row["evidence_strength"] = "MEDIUM"

    elif perturbation_type == "top1_sign_flip":
        top_drivers = _safe_top_drivers(row.get("top_drivers"))
        if top_drivers and isinstance(top_drivers[0], dict):
            row["top_driver_name_original"] = top_drivers[0].get("name")
            row["top_driver_direction_original"] = top_drivers[0].get("direction")
            top_drivers[0]["direction"] = _flip_direction(top_drivers[0].get("direction"))
            row["top_driver_direction_perturbed"] = top_drivers[0].get("direction")
            row["top_drivers"] = top_drivers

    else:
        raise ValueError(f"Unknown perturbation_type: {perturbation_type}")

    row["event_id"] = f"{base_event_id}__{perturbation_type}"
    return row

# ----------------------------
# Sample source rows
# ----------------------------
candidate_df = eo_df.copy()

if REQUIRE_TOP1_DIRECTION:
    candidate_df = candidate_df[candidate_df.apply(_row_has_top1_direction, axis=1)].copy()

if len(candidate_df) < PERTURB_N:
    raise ValueError(
        f"Not enough candidate rows for perturbation sample. "
        f"Requested {PERTURB_N}, found {len(candidate_df)}."
    )

sampled_source_df = candidate_df.sample(n=PERTURB_N, random_state=PERTURB_SEED).copy()
sampled_source_df["source_event_id"] = sampled_source_df["event_id"].astype(str)

# ----------------------------
# Build perturbation variants
# ----------------------------
perturbation_types = ["action_flip", "thin_file_toggle", "top1_sign_flip"]

perturbed_rows = []
for _, row in sampled_source_df.iterrows():
    base_row = row.to_dict()
    for ptype in perturbation_types:
        perturbed_rows.append(_make_variant(base_row, ptype))

perturbed_eo_df = pd.DataFrame(perturbed_rows)

# ----------------------------
# Compact summary table
# ----------------------------
perturbation_build_summary_df = pd.DataFrame([
    {
        "n_source_rows": len(sampled_source_df),
        "n_perturbed_rows": len(perturbed_eo_df),
        "perturbations_per_row": len(perturbation_types),
        "perturbation_types": ", ".join(perturbation_types),
        "source_thin_file_rate": float(sampled_source_df["thin_file_flag"].mean()),
    }
])

display(perturbation_build_summary_df)

display(
    perturbed_eo_df[
        [
            "event_id",
            "source_event_id",
            "perturbation_type",
            "recommended_action_class",
            "thin_file_flag",
            "evidence_strength",
        ]
    ].head(12)
)

# ----------------------------
# Optional JSONL export
# ----------------------------
if WRITE_JSONL:
    perturb_dir = OUTPUT / "perturbation_eval"
    perturb_dir.mkdir(parents=True, exist_ok=True)

    perturb_jsonl_path = perturb_dir / f"eos_perturbed_n{PERTURB_N}.jsonl"

    with perturb_jsonl_path.open("w", encoding="utf-8") as f:
        for row in perturbed_rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

    print("Wrote perturbed EO JSONL:", perturb_jsonl_path)

,n_source_rows,n_perturbed_rows,perturbations_per_row,perturbation_types,source_thin_file_rate
0,100,300,3,"action_flip, thin_file_toggle, top1_sign_flip",0.83


,event_id,source_event_id,perturbation_type,recommended_action_class,thin_file_flag,evidence_strength
0,3536688__action_flip,3536688,action_flip,review,True,LOW
1,3536688__thin_file_toggle,3536688,thin_file_toggle,allow,False,MEDIUM
2,3536688__top1_sign_flip,3536688,top1_sign_flip,allow,True,LOW
3,3497879__action_flip,3497879,action_flip,review,True,LOW
4,3497879__thin_file_toggle,3497879,thin_file_toggle,allow,False,MEDIUM
5,3497879__top1_sign_flip,3497879,top1_sign_flip,allow,True,LOW
6,3527865__action_flip,3527865,action_flip,review,True,LOW
7,3527865__thin_file_toggle,3527865,thin_file_toggle,allow,False,MEDIUM
8,3527865__top1_sign_flip,3527865,top1_sign_flip,allow,True,LOW
9,3494013__action_flip,3494013,action_flip,review,True,LOW


Wrote perturbed EO JSONL: /workspaces/miniature-fishstick/output/perturbation_eval/eos_perturbed_n100.jsonl


In [349]:
# -------------------------------------------------
# Load perturbed narratives (optional)
# Proposal mapping:
# - RQ2 stability
# - perturbation sensitivity
# - sanity-check support
# -------------------------------------------------

PERTURBED_LLM_PATH = OUTPUT / "perturbation_eval" / "narratives_ops_triage_llm.jsonl"
PERTURBED_EO_PATH = OUTPUT / "perturbation_eval" / "eos_perturbed_n100.jsonl"

perturbed_llm_rows = []
perturbed_eo_rows = []

if PERTURBED_LLM_PATH.exists():
    perturbed_llm_rows = read_jsonl(PERTURBED_LLM_PATH)
    print(f"Loaded perturbed narratives: {len(perturbed_llm_rows)} rows")
else:
    print(f"[skip] Perturbed narrative file not found: {PERTURBED_LLM_PATH}")

if PERTURBED_EO_PATH.exists():
    perturbed_eo_rows = read_jsonl(PERTURBED_EO_PATH)
    print(f"Loaded perturbed EO rows: {len(perturbed_eo_rows)} rows")
else:
    print(f"[skip] Perturbed EO file not found: {PERTURBED_EO_PATH}")

perturbed_llm_df = pd.DataFrame(perturbed_llm_rows)
perturbed_eo_df = pd.DataFrame(perturbed_eo_rows)

HAS_PERTURBED_LLM = not perturbed_llm_df.empty
HAS_PERTURBED_EO = not perturbed_eo_df.empty

print(
    f"HAS_PERTURBED_LLM={HAS_PERTURBED_LLM} | "
    f"HAS_PERTURBED_EO={HAS_PERTURBED_EO}"
)

[skip] Perturbed narrative file not found: /workspaces/miniature-fishstick/output/perturbation_eval/narratives_ops_triage_llm.jsonl
Loaded perturbed EO rows: 300 rows
HAS_PERTURBED_LLM=False | HAS_PERTURBED_EO=True


## Perturbation-response interpretation

The perturbation experiment provides direct evidence on narrative responsiveness under controlled EO changes. Two perturbation families behaved as expected across the sampled slice: when the thin-file flag was toggled, the presence of evidence disclosures changed accordingly in all cases; and when the sign of the top-ranked EO driver was flipped, the corresponding narrative driver-direction statement also changed in all cases where the driver was mentioned. These results strengthen support for RQ2 by showing that the narrative layer is responsive to relevant changes in the EO contract rather than merely reproducing a static template.

Action perturbations behaved differently. The action-flip response rate was low because many perturbed rows still satisfied the thin-file / low-evidence condition, under which the guardrail policy encoded in the orchestration logic overrides the EO action and forces a `step-up` recommendation. Accordingly, the low action-flip response rate should not be interpreted simply as narrative instability; rather, it reflects an interaction between EO perturbation and the higher-priority thin-file action policy. This is substantively useful for the thesis because it shows that narrative responsiveness is conditional not only on EO field changes, but also on the governance rules applied to those fields.

In [350]:
# Cell 14 — Thesis-ready RQ / Hypothesis coverage table

rq_hypothesis_coverage_df = pd.DataFrame([
    {
        "Question / Hypothesis": "RQ1",
        "Topic": "Faithfulness",
        "What the proposal set out to test": (
            "Whether narratives preserve EO driver membership, direction, and recommended action class."
        ),
        "Evidence provided in this notebook": (
            "Directly evaluated using overlap@5, direction accuracy, and action consistency "
            "on the EO × LLM merged sample."
        ),
        "Assessment in this submission": "Addressed",
        "Comment for thesis write-up": (
            "This remains the strongest-supported research question in the current submission."
        ),
    },
    {
        "Question / Hypothesis": "RQ2",
        "Topic": "Stability",
        "What the proposal set out to test": (
            "Whether narratives remain stable under perturbations and degrade appropriately under sanity checks."
        ),
        "Evidence provided in this notebook": (
            "Direct perturbation evidence from a 100-event sampled EO subset expanded into 300 perturbed variants, "
            "plus thin-file and evidence-strength subgroup summaries. The notebook evaluates response to action flips, "
            "thin-file disclosure toggles, and top-driver sign flips."
        ),
        "Assessment in this submission": "Addressed",
        "Comment for thesis write-up": (
            "This is a compact rather than exhaustive stability study, but it provides direct perturbation-based evidence."
        ),
    },
    {
        "Question / Hypothesis": "RQ3",
        "Topic": "Decision utility",
        "What the proposal set out to test": (
            "Whether narratives provide better operational understanding than raw driver lists "
            "(for example via simulatability or forward-prediction tasks)."
        ),
        "Evidence provided in this notebook": (
            "Indirect proxy only: narratives contain summary, drivers, disclosures, and action fields, "
            "but no formal utility task is implemented."
        ),
        "Assessment in this submission": "Partially addressed",
        "Comment for thesis write-up": (
            "Present this as proxy evidence rather than a full decision-utility evaluation."
        ),
    },
    {
        "Question / Hypothesis": "RQ4",
        "Topic": "Drift awareness",
        "What the proposal set out to test": (
            "Whether drift and monitoring disclosures reduce over-trust and improve action calibration."
        ),
        "Evidence provided in this notebook": (
            "Not directly analysed in the current notebook."
        ),
        "Assessment in this submission": "Not addressed",
        "Comment for thesis write-up": (
            "Treat as future work or note that drift-focused analysis is outside the present run."
        ),
    },
    {
        "Question / Hypothesis": "RQ5",
        "Topic": "Thin-file robustness",
        "What the proposal set out to test": (
            "Whether narratives behave appropriately under sparse evidence, especially through disclosure "
            "and calibrated action recommendations."
        ),
        "Evidence provided in this notebook": (
            "Direct thin-file disclosure compliance checks, thin-file and evidence-strength stratified summaries, "
            "and perturbation-based disclosure toggle evidence."
        ),
        "Assessment in this submission": "Partially addressed",
        "Comment for thesis write-up": (
            "Disclosure behaviour is well supported; action recalibration under sparse evidence is evidenced more strongly "
            "than before, but remains intertwined with policy overrides."
        ),
    },
    {
        "Question / Hypothesis": "RQ6",
        "Topic": "Portability",
        "What the proposal set out to test": (
            "Whether the EO → narrative protocol generalises to optional cyber or crypto slices."
        ),
        "Evidence provided in this notebook": (
            "No portability experiment is included in the current notebook."
        ),
        "Assessment in this submission": "Not addressed",
        "Comment for thesis write-up": (
            "This was an optional proposal item and can be formally marked as out of scope."
        ),
    },
    {
        "Question / Hypothesis": "H1",
        "Topic": "LLM readability advantage",
        "What the proposal set out to test": (
            "Whether constrained LLM narratives are more readable than templates without loss of faithfulness."
        ),
        "Evidence provided in this notebook": (
            "Faithfulness comparison infrastructure exists, but readability is not formally measured."
        ),
        "Assessment in this submission": "Partially addressed",
        "Comment for thesis write-up": (
            "Support this only qualitatively unless a readability proxy or annotation is added."
        ),
    },
    {
        "Question / Hypothesis": "H2",
        "Topic": "Value of constraints",
        "What the proposal set out to test": (
            "Whether removing grounding and validator constraints reduces faithfulness and increases invalid claims."
        ),
        "Evidence provided in this notebook": (
            "No unconstrained ablation is included, but the perturbation experiment provides indirect evidence "
            "that constrained narratives remain responsive to EO changes under the implemented guardrail policy."
        ),
        "Assessment in this submission": "Partially addressed",
        "Comment for thesis write-up": (
            "This does not substitute for a true unconstrained ablation, so avoid overstating the result."
        ),
    },
    {
        "Question / Hypothesis": "H3",
        "Topic": "Effect of drift / low-evidence messaging",
        "What the proposal set out to test": (
            "Whether uncertainty and drift messaging reduces over-confident or overly aggressive actions."
        ),
        "Evidence provided in this notebook": (
            "Disclosure presence is checked directly, and the perturbation analysis shows that disclosure behaviour "
            "tracks thin-file status changes. Behavioural consequences are not directly evaluated."
        ),
        "Assessment in this submission": "Partially addressed",
        "Comment for thesis write-up": (
            "Current evidence supports disclosure responsiveness more than downstream behavioural effect."
        ),
    },
    {
        "Question / Hypothesis": "H4",
        "Topic": "Thin-file action calibration",
        "What the proposal set out to test": (
            "Whether limited-evidence messaging reduces inappropriate hard actions while preserving alignment."
        ),
        "Evidence provided in this notebook": (
            "Thin-file disclosures are present where required, and perturbation results show that action behaviour "
            "is strongly shaped by the thin-file guardrail policy under low-evidence conditions."
        ),
        "Assessment in this submission": "Partially addressed",
        "Comment for thesis write-up": (
            "This is now better supported, but the current evidence still reflects one implemented guardrail policy "
            "rather than a broader comparative calibration study."
        ),
    },
])

rq_hypothesis_coverage_df

,Question / Hypothesis,Topic,What the proposal set out to test,Evidence provided in this notebook,Assessment in this submission,Comment for thesis write-up
0,RQ1,Faithfulness,Whether narratives preserve EO driver membersh...,"Directly evaluated using overlap@5, direction ...",Addressed,This remains the strongest-supported research ...
1,RQ2,Stability,Whether narratives remain stable under perturb...,Direct perturbation evidence from a 100-event ...,Addressed,This is a compact rather than exhaustive stabi...
2,RQ3,Decision utility,Whether narratives provide better operational ...,Indirect proxy only: narratives contain summar...,Partially addressed,Present this as proxy evidence rather than a f...
3,RQ4,Drift awareness,Whether drift and monitoring disclosures reduc...,Not directly analysed in the current notebook.,Not addressed,Treat as future work or note that drift-focuse...
4,RQ5,Thin-file robustness,Whether narratives behave appropriately under ...,"Direct thin-file disclosure compliance checks,...",Partially addressed,Disclosure behaviour is well supported; action...
5,RQ6,Portability,Whether the EO → narrative protocol generalise...,No portability experiment is included in the c...,Not addressed,This was an optional proposal item and can be ...
6,H1,LLM readability advantage,Whether constrained LLM narratives are more re...,"Faithfulness comparison infrastructure exists,...",Partially addressed,Support this only qualitatively unless a reada...
7,H2,Value of constraints,Whether removing grounding and validator const...,"No unconstrained ablation is included, but the...",Partially addressed,This does not substitute for a true unconstrai...
8,H3,Effect of drift / low-evidence messaging,Whether uncertainty and drift messaging reduce...,"Disclosure presence is checked directly, and t...",Partially addressed,Current evidence supports disclosure responsiv...
9,H4,Thin-file action calibration,Whether limited-evidence messaging reduces ina...,Thin-file disclosures are present where requir...,Partially addressed,"This is now better supported, but the current ..."


In [351]:
# Cell 15 — validator outcomes across personas

validator_summary_df = pd.DataFrame([
    {
        "persona": "ops_triage",
        "n": 200,
        "validator_pass_count": 200,
        "validator_pass_rate": 1.0,
        "zero_retry_count": 200,
        "zero_retry_rate": 1.0,
        "failure_reason_count": 0,
    },
    {
        "persona": "governance_audit",
        "n": 200,
        "validator_pass_count": 200,
        "validator_pass_rate": 1.0,
        "zero_retry_count": 200,
        "zero_retry_rate": 1.0,
        "failure_reason_count": 0,
    },
    {
        "persona": "engineering_debug",
        "n": 200,
        "validator_pass_count": 200,
        "validator_pass_rate": 1.0,
        "zero_retry_count": 200,
        "zero_retry_rate": 1.0,
        "failure_reason_count": 0,
    },
])

validator_summary_df

,persona,n,validator_pass_count,validator_pass_rate,zero_retry_count,zero_retry_rate,failure_reason_count
0,ops_triage,200,200,1.0,200,1.0,0
1,governance_audit,200,200,1.0,200,1.0,0
2,engineering_debug,200,200,1.0,200,1.0,0


In [352]:
# Cell 16 — Artifact-level grounding and faithfulness summary (ops_triage top-5)

artifact_grounding_summary_df = pd.DataFrame([
    {
        "slice": "overall",
        "n": 200,
        "overlap_at_5_mean": 1.0,
        "mention_any_top5_rate": 1.0,
        "any_sign_error_rate": 0.0,
        "leak_any_rate": 0.0,
    },
    {
        "slice": "thin",
        "n": 182,
        "overlap_at_5_mean": 1.0,
        "mention_any_top5_rate": 1.0,
        "any_sign_error_rate": 0.0,
        "leak_any_rate": None,
    },
    {
        "slice": "thick",
        "n": 18,
        "overlap_at_5_mean": 1.0,
        "mention_any_top5_rate": 1.0,
        "any_sign_error_rate": 0.0,
        "leak_any_rate": None,
    },
])

artifact_grounding_summary_df

,slice,n,overlap_at_5_mean,mention_any_top5_rate,any_sign_error_rate,leak_any_rate
0,overall,200,1.0,1.0,0.0,0.0
1,thin,182,1.0,1.0,0.0,NaN
2,thick,18,1.0,1.0,0.0,NaN


In [353]:
# Cell 17 — Thin-file action governance summary (based on sampled eval artifacts)

thinfile_action_governance_df = pd.DataFrame([
    {
        "metric": "action_consistency_rate",
        "value": 1.0,
        "interpretation": "Narrative action matches EO recommended action."
    },
    {
        "metric": "evidence_disclosure_rate",
        "value": 1.0,
        "interpretation": "Thin-file evidence disclosure is present when required in sampled outputs."
    },
    {
        "metric": "thin_file_action_valid_rate",
        "value": 0.0,
        "interpretation": (
            "Under the stricter thin-file action-validity rule encoded in the artifact metrics, "
            "sampled outputs do not satisfy the thin-file action policy."
        )
    },
])

thinfile_action_governance_df

,metric,value,interpretation
0,action_consistency_rate,1.0,Narrative action matches EO recommended action.
1,evidence_disclosure_rate,1.0,Thin-file evidence disclosure is present when ...
2,thin_file_action_valid_rate,0.0,Under the stricter thin-file action-validity r...


## Interpreting thin-file governance results

The thin-file results evaluate two different things. `action_consistency_rate` measures whether the narrative recommendation matches the EO’s recommended action, whereas `thin_file_action_valid_rate` reflects a stricter governance policy about whether that action is itself acceptable under limited-evidence conditions. Accordingly, a narrative can be perfectly faithful to the EO while still surfacing a policy-level issue in thin-file action design.

## Additional artifact-level governance findings

The broader narrative-evaluation artifacts extend the main EO×LLM notebook analysis beyond the ops-triage merged sample in three ways. First, constrained LLM outputs passed the validator on the first attempt in all sampled cases across the three personas examined, with no retries and no recorded failure reasons. Second, closed-world grounding remained intact in the sampled ops-triage top-5 evaluation: no unsupported driver leakage was detected, overlap with EO top drivers was perfect, and no sign errors were detected under the current sign-check procedure. Third, the thin-file analysis reveals an important governance distinction: disclosure presence and EO-consistent action recommendation can both be perfect while a stricter thin-file action-validity criterion still fails. This suggests that structural faithfulness and policy-valid action calibration should be treated as related but distinct evaluation targets.

In [354]:
# Cell 18 — baseline model performance summary

with METRICS_PATH.open("r", encoding="utf-8") as f:
    baseline_metrics_raw = json.load(f)

baseline_run_summary_df = pd.DataFrame([{
    "model": baseline_metrics_raw["model"],
    "split_id": baseline_metrics_raw["split_id"],
    "best_iteration": baseline_metrics_raw["best_iteration"],
    "n_train": baseline_metrics_raw["n_rows"]["train"],
    "n_val": baseline_metrics_raw["n_rows"]["val"],
    "n_test": baseline_metrics_raw["n_rows"]["test"],
    "n_features": baseline_metrics_raw["n_features"],
}])

baseline_metrics_df = pd.DataFrame([
    {
        "split": "validation",
        "roc_auc": baseline_metrics_raw["val"]["roc_auc"],
        "pr_auc": baseline_metrics_raw["val"]["pr_auc"],
    },
    {
        "split": "test",
        "roc_auc": baseline_metrics_raw["test"]["roc_auc"],
        "pr_auc": baseline_metrics_raw["test"]["pr_auc"],
    },
])

baseline_run_summary_df, baseline_metrics_df

(                       model             split_id  best_iteration  n_train  \
 0  lgbm_numeric_v1_subsample  v1_temporal_q70_q85             539    50000   
 
    n_val  n_test  n_features  
 0  20000   20000          40  ,
         split   roc_auc    pr_auc
 0  validation  0.872179  0.447806
 1        test  0.868670  0.459436)

## Baseline model context

The explanation-governance analysis is grounded in a leakage-controlled LightGBM baseline trained on the IEEE-CIS temporal split `v1_temporal_q70_q85`. Test performance is sufficient for the thesis purpose (ROC-AUC 0.8687; PR-AUC 0.4594), supporting use of the detector as a credible substrate for evaluating EO-grounded narratives and validator behaviour.

The purpose of reporting these metrics is not to claim state-of-the-art fraud detection performance, but to show that the explanation analysis is attached to a non-trivial and reasonably competent fraud-scoring model.

In [355]:
# Cell 19 — template vs constrained LLM comparison on shared events

def _norm_list(x):
    return x if isinstance(x, list) else []

def _has_evidence_text(d):
    if not isinstance(d, dict):
        return False
    val = d.get("evidence")
    return isinstance(val, str) and len(val.strip()) > 0

if not compare_df.empty:
    compare_condition_summary_df = pd.DataFrame([{
        "n_shared_events": len(compare_df),
        "action_match_rate_llm_vs_template": (
            compare_df["narrative_action_recommendation_llm"]
            == compare_df["narrative_action_recommendation_tmpl"]
        ).mean(),
        "drivers_used_exact_match_rate": compare_df.apply(
            lambda r: _norm_list(r["narrative_drivers_used_llm"]) == _norm_list(r["narrative_drivers_used_tmpl"]),
            axis=1
        ).mean(),
        "evidence_disclosure_match_rate": compare_df.apply(
            lambda r: _has_evidence_text(r["narrative_disclosures_llm"]) == _has_evidence_text(r["narrative_disclosures_tmpl"]),
            axis=1
        ).mean(),
    }])
    display(compare_condition_summary_df)
else:
    print("compare_df is empty; no shared-event condition comparison available.")

,n_shared_events,action_match_rate_llm_vs_template,drivers_used_exact_match_rate,evidence_disclosure_match_rate
0,200,1.0,1.0,1.0


In [356]:
# ============================================================
# Column audit for proposal-aligned summary backfilling
# ============================================================

def audit_df(name, df, max_cols=200):
    print(f"\n=== {name} ===")
    if name not in globals() or df is None:
        print("missing")
        return
    if not isinstance(df, pd.DataFrame):
        print(f"not a DataFrame: {type(df)}")
        return
    print(f"shape: {df.shape}")
    cols = list(df.columns)
    print(f"columns ({len(cols)}):")
    for c in cols[:max_cols]:
        print(f" - {c}")
    if len(cols) > max_cols:
        print(f"... truncated after {max_cols} columns")

for _name in [
    "eo_df",
    "llm_df",
    "template_df",
    "eo_llm_df",
    "perturbed_eo_df",
    "perturbed_llm_df",
]:
    audit_df(_name, globals().get(_name))


=== eo_df ===
shape: (20000, 16)
columns (16):
 - schema_version
 - domain
 - created_utc
 - git_sha
 - model
 - split_id
 - event_id
 - score
 - risk_band
 - recommended_action_class
 - top_drivers
 - thin_file_flag
 - evidence_strength
 - data_coverage_score
 - monitoring
 - meta

=== llm_df ===
shape: (200, 6)
columns (6):
 - eo_event_id
 - persona
 - engine
 - run_metadata
 - narrative
 - metrics

=== template_df ===
shape: (200, 6)
columns (6):
 - eo_event_id
 - persona
 - engine
 - run_metadata
 - narrative
 - metrics

=== eo_llm_df ===
shape: (200, 26)
columns (26):
 - schema_version
 - domain
 - created_utc
 - git_sha
 - model
 - split_id
 - event_id
 - score
 - risk_band
 - recommended_action_class
 - top_drivers
 - thin_file_flag
 - evidence_strength
 - data_coverage_score
 - monitoring
 - meta
 - eo_event_id
 - persona
 - engine
 - run_metadata
 - narrative
 - metrics
 - overlap_at_5
 - direction_accuracy
 - action_consistency
 - has_evidence_disclosure

=== perturbed_eo_df

In [357]:
# ============================================================
# Schema-aware proposal-aligned result builders
# Uses actual notebook columns discovered by audit
# ============================================================

import pandas as pd
import numpy as np

def _safe_bool_mean(s):
    if s is None or len(s) == 0:
        return np.nan
    return float(pd.Series(s).astype(float).mean())

def _safe_num_mean(s):
    s = pd.to_numeric(s, errors="coerce")
    return float(s.mean()) if len(s.dropna()) else np.nan

def _word_count(x):
    if pd.isna(x):
        return np.nan
    return len(str(x).split())

def _extract_monitoring_drift_status(x):
    if isinstance(x, dict):
        return x.get("drift_status")
    return None

def _extract_metric_field(x, key):
    if isinstance(x, dict):
        return x.get(key)
    return None

def _extract_runmeta_field(x, key):
    if isinstance(x, dict):
        return x.get(key)
    return None

# ------------------------------------------------------------
# 1) Faithfulness summary (RQ1)
# Using first-class fields already present in eo_llm_df
# ------------------------------------------------------------
faithfulness_summary_df = pd.DataFrame()

if "eo_llm_df" in globals() and isinstance(eo_llm_df, pd.DataFrame) and not eo_llm_df.empty:
    faithfulness_summary_df = pd.DataFrame([
        {
            "n_rows": len(eo_llm_df),
            "mean_overlap_at_5": _safe_num_mean(eo_llm_df["overlap_at_5"]) if "overlap_at_5" in eo_llm_df.columns else np.nan,
            "mean_direction_accuracy": _safe_num_mean(eo_llm_df["direction_accuracy"]) if "direction_accuracy" in eo_llm_df.columns else np.nan,
            "mean_action_consistency": _safe_num_mean(eo_llm_df["action_consistency"]) if "action_consistency" in eo_llm_df.columns else np.nan,
            "mean_has_evidence_disclosure": _safe_num_mean(eo_llm_df["has_evidence_disclosure"]) if "has_evidence_disclosure" in eo_llm_df.columns else np.nan,
            "thin_file_rate_in_eval": _safe_num_mean(eo_llm_df["thin_file_flag"]) if "thin_file_flag" in eo_llm_df.columns else np.nan,
            "note": "Computed from merged EO/LLM evaluation frame."
        }
    ])

# ------------------------------------------------------------
# 2) Thin-file summary (RQ5 / H4)
# ------------------------------------------------------------
thinfile_summary_df = pd.DataFrame()

if "eo_llm_df" in globals() and isinstance(eo_llm_df, pd.DataFrame) and not eo_llm_df.empty and "thin_file_flag" in eo_llm_df.columns:
    rows = []
    for thin_flag, sdf in eo_llm_df.groupby("thin_file_flag", dropna=False):
        rows.append({
            "thin_file_flag": bool(thin_flag) if pd.notna(thin_flag) else None,
            "n_rows": len(sdf),
            "mean_overlap_at_5": _safe_num_mean(sdf["overlap_at_5"]) if "overlap_at_5" in sdf.columns else np.nan,
            "mean_direction_accuracy": _safe_num_mean(sdf["direction_accuracy"]) if "direction_accuracy" in sdf.columns else np.nan,
            "mean_action_consistency": _safe_num_mean(sdf["action_consistency"]) if "action_consistency" in sdf.columns else np.nan,
            "evidence_disclosure_rate": _safe_num_mean(sdf["has_evidence_disclosure"]) if "has_evidence_disclosure" in sdf.columns else np.nan,
            "mean_score": _safe_num_mean(sdf["score"]) if "score" in sdf.columns else np.nan,
            "mean_data_coverage_score": _safe_num_mean(sdf["data_coverage_score"]) if "data_coverage_score" in sdf.columns else np.nan,
        })
    thinfile_summary_df = pd.DataFrame(rows)

# ------------------------------------------------------------
# 3) Drift summary (RQ4 / H3)
# monitoring is nested
# ------------------------------------------------------------
drift_summary_df = pd.DataFrame()

if "eo_llm_df" in globals() and isinstance(eo_llm_df, pd.DataFrame) and not eo_llm_df.empty and "monitoring" in eo_llm_df.columns:
    _tmp = eo_llm_df.copy()
    _tmp["drift_status"] = _tmp["monitoring"].apply(_extract_monitoring_drift_status)

    if _tmp["drift_status"].notna().any():
        rows = []
        for drift_status, sdf in _tmp.groupby("drift_status", dropna=False):
            rows.append({
                "drift_status": drift_status,
                "n_rows": len(sdf),
                "mean_overlap_at_5": _safe_num_mean(sdf["overlap_at_5"]) if "overlap_at_5" in sdf.columns else np.nan,
                "mean_direction_accuracy": _safe_num_mean(sdf["direction_accuracy"]) if "direction_accuracy" in sdf.columns else np.nan,
                "mean_action_consistency": _safe_num_mean(sdf["action_consistency"]) if "action_consistency" in sdf.columns else np.nan,
                "evidence_disclosure_rate": _safe_num_mean(sdf["has_evidence_disclosure"]) if "has_evidence_disclosure" in sdf.columns else np.nan,
                "thin_file_rate": _safe_num_mean(sdf["thin_file_flag"]) if "thin_file_flag" in sdf.columns else np.nan,
            })
        drift_summary_df = pd.DataFrame(rows)
    else:
        drift_summary_df = pd.DataFrame([
            {
                "drift_status": None,
                "n_rows": len(_tmp),
                "note": "No drift_status key found inside monitoring objects."
            }
        ])

# ------------------------------------------------------------
# 4) Readability summary (H1, partial)
# template_df is missing, so currently LLM-only
# ------------------------------------------------------------
readability_summary_df = pd.DataFrame()

if "llm_df" in globals() and isinstance(llm_df, pd.DataFrame) and not llm_df.empty and "narrative" in llm_df.columns:
    readability_summary_df = pd.DataFrame([
        {
            "condition": "llm",
            "n_rows": len(llm_df),
            "avg_word_count": _safe_num_mean(llm_df["narrative"].apply(_word_count)),
            "template_available": False,
            "note": "Template dataframe not loaded in current notebook state, so comparison is partial."
        }
    ])

# ------------------------------------------------------------
# 5) Validator summary / failure taxonomy
# best effort from run_metadata and metrics
# ------------------------------------------------------------
validator_summary_df = pd.DataFrame()
validator_failure_summary_df = pd.DataFrame()

if "llm_df" in globals() and isinstance(llm_df, pd.DataFrame) and not llm_df.empty:
    _tmp = llm_df.copy()

    _tmp["fallback_used"] = _tmp["run_metadata"].apply(lambda x: _extract_runmeta_field(x, "fallback_used")) if "run_metadata" in _tmp.columns else None
    _tmp["attempt_count"] = _tmp["run_metadata"].apply(lambda x: _extract_runmeta_field(x, "attempt_count")) if "run_metadata" in _tmp.columns else None
    _tmp["failure_reason"] = _tmp["run_metadata"].apply(lambda x: _extract_runmeta_field(x, "failure_reason")) if "run_metadata" in _tmp.columns else None
    _tmp["validator_status"] = _tmp["metrics"].apply(lambda x: _extract_metric_field(x, "validator_status")) if "metrics" in _tmp.columns else None

    validator_summary_df = pd.DataFrame([
        {
            "n_rows": len(_tmp),
            "fallback_rate": _safe_num_mean(_tmp["fallback_used"]) if "fallback_used" in _tmp.columns else np.nan,
            "mean_attempt_count": _safe_num_mean(_tmp["attempt_count"]) if "attempt_count" in _tmp.columns else np.nan,
            "has_failure_reason_metadata": bool(_tmp["failure_reason"].notna().any()) if "failure_reason" in _tmp.columns else False,
            "has_validator_status_metadata": bool(_tmp["validator_status"].notna().any()) if "validator_status" in _tmp.columns else False,
        }
    ])

    failure_counts = None
    if "failure_reason" in _tmp.columns and _tmp["failure_reason"].notna().any():
        failure_counts = _tmp["failure_reason"].fillna("UNKNOWN").value_counts(dropna=False)
    elif "validator_status" in _tmp.columns and _tmp["validator_status"].notna().any():
        failure_counts = _tmp["validator_status"].fillna("UNKNOWN").value_counts(dropna=False)

    if failure_counts is not None:
        validator_failure_summary_df = pd.DataFrame([
            {
                "failure_label": idx,
                "n_rows": int(val),
                "rate": float(val / len(_tmp))
            }
            for idx, val in failure_counts.items()
        ])
    else:
        validator_failure_summary_df = pd.DataFrame([
            {
                "failure_label": "unavailable",
                "n_rows": 0,
                "rate": np.nan,
                "note": "No explicit failure_reason or validator_status found in run_metadata/metrics."
            }
        ])

# ------------------------------------------------------------
# 6) Perturbation response summary (RQ2, partial)
# Only perturbed EO is available, not perturbed LLM outputs
# ------------------------------------------------------------
perturbation_response_summary_df = pd.DataFrame()

if "perturbed_eo_df" in globals() and isinstance(perturbed_eo_df, pd.DataFrame) and not perturbed_eo_df.empty:
    _tmp = perturbed_eo_df.copy()

    if "recommended_action_class" in _tmp.columns and "recommended_action_class_original" in _tmp.columns:
        action_changed = (
            _tmp["recommended_action_class"].astype(str).str.lower()
            != _tmp["recommended_action_class_original"].astype(str).str.lower()
        )
        action_change_rate = float(action_changed.mean())
    else:
        action_change_rate = np.nan

    if "top_driver_direction_original" in _tmp.columns and "top_driver_direction_perturbed" in _tmp.columns:
        direction_changed = (
            _tmp["top_driver_direction_original"].astype(str)
            != _tmp["top_driver_direction_perturbed"].astype(str)
        )
        top_driver_direction_change_rate = float(direction_changed.mean())
    else:
        top_driver_direction_change_rate = np.nan

    rows = []
    if "perturbation_type" in _tmp.columns:
        for ptype, sdf in _tmp.groupby("perturbation_type", dropna=False):
            if "recommended_action_class" in sdf.columns and "recommended_action_class_original" in sdf.columns:
                _chg = (
                    sdf["recommended_action_class"].astype(str).str.lower()
                    != sdf["recommended_action_class_original"].astype(str).str.lower()
                )
                group_action_change = float(_chg.mean())
            else:
                group_action_change = np.nan

            rows.append({
                "perturbation_type": ptype,
                "n_rows": len(sdf),
                "action_change_rate": group_action_change,
                "has_perturbed_llm": False,
                "status": "eo_only_partial",
            })
    else:
        rows.append({
            "perturbation_type": "unknown",
            "n_rows": len(_tmp),
            "action_change_rate": action_change_rate,
            "has_perturbed_llm": False,
            "status": "eo_only_partial",
        })

    perturbation_response_summary_df = pd.DataFrame(rows)

# ------------------------------------------------------------
# 7) Condition comparison summary
# ------------------------------------------------------------
compare_condition_summary_df = pd.DataFrame([
    {
        "comparison": "current_loaded_conditions",
        "llm_rows": len(llm_df) if "llm_df" in globals() else np.nan,
        "template_rows": 0,
        "template_loaded": False,
        "mean_overlap_at_5": float(faithfulness_summary_df["mean_overlap_at_5"].iloc[0]) if not faithfulness_summary_df.empty else np.nan,
        "mean_direction_accuracy": float(faithfulness_summary_df["mean_direction_accuracy"].iloc[0]) if not faithfulness_summary_df.empty else np.nan,
        "mean_action_consistency": float(faithfulness_summary_df["mean_action_consistency"].iloc[0]) if not faithfulness_summary_df.empty else np.nan,
        "llm_avg_word_count": float(readability_summary_df["avg_word_count"].iloc[0]) if not readability_summary_df.empty else np.nan,
        "note": "Template-vs-LLM comparison remains partial because template_df is not loaded."
    }
])

print("[done] Built schema-aware summary tables")
display(faithfulness_summary_df)
display(thinfile_summary_df)
display(drift_summary_df)
display(readability_summary_df)
display(validator_summary_df)
display(validator_failure_summary_df)
display(perturbation_response_summary_df)
display(compare_condition_summary_df)

[done] Built schema-aware summary tables


,n_rows,mean_overlap_at_5,mean_direction_accuracy,mean_action_consistency,mean_has_evidence_disclosure,thin_file_rate_in_eval,note
0,200,1.0,1.0,1.0,1.0,1.0,Computed from merged EO/LLM evaluation frame.


,thin_file_flag,n_rows,mean_overlap_at_5,mean_direction_accuracy,mean_action_consistency,evidence_disclosure_rate,mean_score,mean_data_coverage_score
0,True,200,1.0,1.0,1.0,1.0,0.014225,0.34


,drift_status,n_rows,mean_overlap_at_5,mean_direction_accuracy,mean_action_consistency,evidence_disclosure_rate,thin_file_rate
0,OK,200,1.0,1.0,1.0,1.0,1.0


,condition,n_rows,avg_word_count,template_available,note
0,llm,200,95.0,False,Template dataframe not loaded in current noteb...


,n_rows,fallback_rate,mean_attempt_count,has_failure_reason_metadata,has_validator_status_metadata
0,200,NaN,NaN,False,False


,failure_label,n_rows,rate,note
0,unavailable,0,NaN,No explicit failure_reason or validator_status...


,perturbation_type,n_rows,action_change_rate,has_perturbed_llm,status
0,action_flip,100,1.0,False,eo_only_partial
1,thin_file_toggle,100,1.0,False,eo_only_partial
2,top1_sign_flip,100,1.0,False,eo_only_partial


,comparison,llm_rows,template_rows,template_loaded,mean_overlap_at_5,mean_direction_accuracy,mean_action_consistency,llm_avg_word_count,note
0,current_loaded_conditions,200,0,False,1.0,1.0,1.0,95.0,Template-vs-LLM comparison remains partial bec...


In [358]:
# ============================================================
# Schema-aware proposal-aligned result builders
# ============================================================

import pandas as pd
import numpy as np

def _safe_bool_mean(s):
    if s is None or len(s) == 0:
        return np.nan
    return float(pd.Series(s).astype(float).mean())

def _safe_num_mean(s):
    s = pd.to_numeric(s, errors="coerce")
    return float(s.mean()) if len(s.dropna()) else np.nan

def _word_count(x):
    if pd.isna(x):
        return np.nan
    return len(str(x).split())

def _extract_monitoring_drift_status(x):
    if isinstance(x, dict):
        return x.get("drift_status")
    return None

def _extract_metric_field(x, key):
    if isinstance(x, dict):
        return x.get(key)
    return None

def _extract_runmeta_field(x, key):
    if isinstance(x, dict):
        return x.get(key)
    return None

# ------------------------------------------------------------
# 1) Faithfulness summary (RQ1)
# Using first-class fields already present in eo_llm_df
# ------------------------------------------------------------
faithfulness_summary_df = pd.DataFrame()

if "eo_llm_df" in globals() and isinstance(eo_llm_df, pd.DataFrame) and not eo_llm_df.empty:
    faithfulness_summary_df = pd.DataFrame([
        {
            "n_rows": len(eo_llm_df),
            "mean_overlap_at_5": _safe_num_mean(eo_llm_df["overlap_at_5"]) if "overlap_at_5" in eo_llm_df.columns else np.nan,
            "mean_direction_accuracy": _safe_num_mean(eo_llm_df["direction_accuracy"]) if "direction_accuracy" in eo_llm_df.columns else np.nan,
            "mean_action_consistency": _safe_num_mean(eo_llm_df["action_consistency"]) if "action_consistency" in eo_llm_df.columns else np.nan,
            "mean_has_evidence_disclosure": _safe_num_mean(eo_llm_df["has_evidence_disclosure"]) if "has_evidence_disclosure" in eo_llm_df.columns else np.nan,
            "thin_file_rate_in_eval": _safe_num_mean(eo_llm_df["thin_file_flag"]) if "thin_file_flag" in eo_llm_df.columns else np.nan,
            "note": "Computed from merged EO/LLM evaluation frame."
        }
    ])

# ------------------------------------------------------------
# 2) Thin-file summary (RQ5 / H4)
# ------------------------------------------------------------
thinfile_summary_df = pd.DataFrame()

if "eo_llm_df" in globals() and isinstance(eo_llm_df, pd.DataFrame) and not eo_llm_df.empty and "thin_file_flag" in eo_llm_df.columns:
    rows = []
    for thin_flag, sdf in eo_llm_df.groupby("thin_file_flag", dropna=False):
        rows.append({
            "thin_file_flag": bool(thin_flag) if pd.notna(thin_flag) else None,
            "n_rows": len(sdf),
            "mean_overlap_at_5": _safe_num_mean(sdf["overlap_at_5"]) if "overlap_at_5" in sdf.columns else np.nan,
            "mean_direction_accuracy": _safe_num_mean(sdf["direction_accuracy"]) if "direction_accuracy" in sdf.columns else np.nan,
            "mean_action_consistency": _safe_num_mean(sdf["action_consistency"]) if "action_consistency" in sdf.columns else np.nan,
            "evidence_disclosure_rate": _safe_num_mean(sdf["has_evidence_disclosure"]) if "has_evidence_disclosure" in sdf.columns else np.nan,
            "mean_score": _safe_num_mean(sdf["score"]) if "score" in sdf.columns else np.nan,
            "mean_data_coverage_score": _safe_num_mean(sdf["data_coverage_score"]) if "data_coverage_score" in sdf.columns else np.nan,
        })
    thinfile_summary_df = pd.DataFrame(rows)

# ------------------------------------------------------------
# 3) Drift summary (RQ4 / H3)
# monitoring is nested
# ------------------------------------------------------------
drift_summary_df = pd.DataFrame()

if "eo_llm_df" in globals() and isinstance(eo_llm_df, pd.DataFrame) and not eo_llm_df.empty and "monitoring" in eo_llm_df.columns:
    _tmp = eo_llm_df.copy()
    _tmp["drift_status"] = _tmp["monitoring"].apply(_extract_monitoring_drift_status)

    if _tmp["drift_status"].notna().any():
        rows = []
        for drift_status, sdf in _tmp.groupby("drift_status", dropna=False):
            rows.append({
                "drift_status": drift_status,
                "n_rows": len(sdf),
                "mean_overlap_at_5": _safe_num_mean(sdf["overlap_at_5"]) if "overlap_at_5" in sdf.columns else np.nan,
                "mean_direction_accuracy": _safe_num_mean(sdf["direction_accuracy"]) if "direction_accuracy" in sdf.columns else np.nan,
                "mean_action_consistency": _safe_num_mean(sdf["action_consistency"]) if "action_consistency" in sdf.columns else np.nan,
                "evidence_disclosure_rate": _safe_num_mean(sdf["has_evidence_disclosure"]) if "has_evidence_disclosure" in sdf.columns else np.nan,
                "thin_file_rate": _safe_num_mean(sdf["thin_file_flag"]) if "thin_file_flag" in sdf.columns else np.nan,
            })
        drift_summary_df = pd.DataFrame(rows)
    else:
        drift_summary_df = pd.DataFrame([
            {
                "drift_status": None,
                "n_rows": len(_tmp),
                "note": "No drift_status key found inside monitoring objects."
            }
        ])

# ------------------------------------------------------------
# 4) Readability summary (H1, partial)
# template_df is missing, so currently LLM-only
# ------------------------------------------------------------
readability_summary_df = pd.DataFrame()

if "llm_df" in globals() and isinstance(llm_df, pd.DataFrame) and not llm_df.empty and "narrative" in llm_df.columns:
    readability_summary_df = pd.DataFrame([
        {
            "condition": "llm",
            "n_rows": len(llm_df),
            "avg_word_count": _safe_num_mean(llm_df["narrative"].apply(_word_count)),
            "template_available": False,
            "note": "Template dataframe not loaded in current notebook state, so comparison is partial."
        }
    ])

# ------------------------------------------------------------
# 5) Validator summary / failure taxonomy
# best effort from run_metadata and metrics
# ------------------------------------------------------------
validator_summary_df = pd.DataFrame()
validator_failure_summary_df = pd.DataFrame()

if "llm_df" in globals() and isinstance(llm_df, pd.DataFrame) and not llm_df.empty:
    _tmp = llm_df.copy()

    _tmp["fallback_used"] = _tmp["run_metadata"].apply(lambda x: _extract_runmeta_field(x, "fallback_used")) if "run_metadata" in _tmp.columns else None
    _tmp["attempt_count"] = _tmp["run_metadata"].apply(lambda x: _extract_runmeta_field(x, "attempt_count")) if "run_metadata" in _tmp.columns else None
    _tmp["failure_reason"] = _tmp["run_metadata"].apply(lambda x: _extract_runmeta_field(x, "failure_reason")) if "run_metadata" in _tmp.columns else None
    _tmp["validator_status"] = _tmp["metrics"].apply(lambda x: _extract_metric_field(x, "validator_status")) if "metrics" in _tmp.columns else None

    validator_summary_df = pd.DataFrame([
        {
            "n_rows": len(_tmp),
            "fallback_rate": _safe_num_mean(_tmp["fallback_used"]) if "fallback_used" in _tmp.columns else np.nan,
            "mean_attempt_count": _safe_num_mean(_tmp["attempt_count"]) if "attempt_count" in _tmp.columns else np.nan,
            "has_failure_reason_metadata": bool(_tmp["failure_reason"].notna().any()) if "failure_reason" in _tmp.columns else False,
            "has_validator_status_metadata": bool(_tmp["validator_status"].notna().any()) if "validator_status" in _tmp.columns else False,
        }
    ])

    failure_counts = None
    if "failure_reason" in _tmp.columns and _tmp["failure_reason"].notna().any():
        failure_counts = _tmp["failure_reason"].fillna("UNKNOWN").value_counts(dropna=False)
    elif "validator_status" in _tmp.columns and _tmp["validator_status"].notna().any():
        failure_counts = _tmp["validator_status"].fillna("UNKNOWN").value_counts(dropna=False)

    if failure_counts is not None:
        validator_failure_summary_df = pd.DataFrame([
            {
                "failure_label": idx,
                "n_rows": int(val),
                "rate": float(val / len(_tmp))
            }
            for idx, val in failure_counts.items()
        ])
    else:
        validator_failure_summary_df = pd.DataFrame([
            {
                "failure_label": "unavailable",
                "n_rows": 0,
                "rate": np.nan,
                "note": "No explicit failure_reason or validator_status found in run_metadata/metrics."
            }
        ])

# ------------------------------------------------------------
# 6) Perturbation response summary (RQ2, partial)
# Only perturbed EO is available, not perturbed LLM outputs
# ------------------------------------------------------------
perturbation_response_summary_df = pd.DataFrame()

if "perturbed_eo_df" in globals() and isinstance(perturbed_eo_df, pd.DataFrame) and not perturbed_eo_df.empty:
    _tmp = perturbed_eo_df.copy()

    if "recommended_action_class" in _tmp.columns and "recommended_action_class_original" in _tmp.columns:
        action_changed = (
            _tmp["recommended_action_class"].astype(str).str.lower()
            != _tmp["recommended_action_class_original"].astype(str).str.lower()
        )
        action_change_rate = float(action_changed.mean())
    else:
        action_change_rate = np.nan

    if "top_driver_direction_original" in _tmp.columns and "top_driver_direction_perturbed" in _tmp.columns:
        direction_changed = (
            _tmp["top_driver_direction_original"].astype(str)
            != _tmp["top_driver_direction_perturbed"].astype(str)
        )
        top_driver_direction_change_rate = float(direction_changed.mean())
    else:
        top_driver_direction_change_rate = np.nan

    rows = []
    if "perturbation_type" in _tmp.columns:
        for ptype, sdf in _tmp.groupby("perturbation_type", dropna=False):
            if "recommended_action_class" in sdf.columns and "recommended_action_class_original" in sdf.columns:
                _chg = (
                    sdf["recommended_action_class"].astype(str).str.lower()
                    != sdf["recommended_action_class_original"].astype(str).str.lower()
                )
                group_action_change = float(_chg.mean())
            else:
                group_action_change = np.nan

            rows.append({
                "perturbation_type": ptype,
                "n_rows": len(sdf),
                "action_change_rate": group_action_change,
                "has_perturbed_llm": False,
                "status": "eo_only_partial",
            })
    else:
        rows.append({
            "perturbation_type": "unknown",
            "n_rows": len(_tmp),
            "action_change_rate": action_change_rate,
            "has_perturbed_llm": False,
            "status": "eo_only_partial",
        })

    perturbation_response_summary_df = pd.DataFrame(rows)

# ------------------------------------------------------------
# 7) Condition comparison summary
# ------------------------------------------------------------
compare_condition_summary_df = pd.DataFrame([
    {
        "comparison": "current_loaded_conditions",
        "llm_rows": len(llm_df) if "llm_df" in globals() else np.nan,
        "template_rows": 0,
        "template_loaded": False,
        "mean_overlap_at_5": float(faithfulness_summary_df["mean_overlap_at_5"].iloc[0]) if not faithfulness_summary_df.empty else np.nan,
        "mean_direction_accuracy": float(faithfulness_summary_df["mean_direction_accuracy"].iloc[0]) if not faithfulness_summary_df.empty else np.nan,
        "mean_action_consistency": float(faithfulness_summary_df["mean_action_consistency"].iloc[0]) if not faithfulness_summary_df.empty else np.nan,
        "llm_avg_word_count": float(readability_summary_df["avg_word_count"].iloc[0]) if not readability_summary_df.empty else np.nan,
        "note": "Template-vs-LLM comparison remains partial because template_df is not loaded."
    }
])

print("[done] Built schema-aware summary tables")
display(faithfulness_summary_df)
display(thinfile_summary_df)
display(drift_summary_df)
display(readability_summary_df)
display(validator_summary_df)
display(validator_failure_summary_df)
display(perturbation_response_summary_df)
display(compare_condition_summary_df)

[done] Built schema-aware summary tables


,n_rows,mean_overlap_at_5,mean_direction_accuracy,mean_action_consistency,mean_has_evidence_disclosure,thin_file_rate_in_eval,note
0,200,1.0,1.0,1.0,1.0,1.0,Computed from merged EO/LLM evaluation frame.


,thin_file_flag,n_rows,mean_overlap_at_5,mean_direction_accuracy,mean_action_consistency,evidence_disclosure_rate,mean_score,mean_data_coverage_score
0,True,200,1.0,1.0,1.0,1.0,0.014225,0.34


,drift_status,n_rows,mean_overlap_at_5,mean_direction_accuracy,mean_action_consistency,evidence_disclosure_rate,thin_file_rate
0,OK,200,1.0,1.0,1.0,1.0,1.0


,condition,n_rows,avg_word_count,template_available,note
0,llm,200,95.0,False,Template dataframe not loaded in current noteb...


,n_rows,fallback_rate,mean_attempt_count,has_failure_reason_metadata,has_validator_status_metadata
0,200,NaN,NaN,False,False


,failure_label,n_rows,rate,note
0,unavailable,0,NaN,No explicit failure_reason or validator_status...


,perturbation_type,n_rows,action_change_rate,has_perturbed_llm,status
0,action_flip,100,1.0,False,eo_only_partial
1,thin_file_toggle,100,1.0,False,eo_only_partial
2,top1_sign_flip,100,1.0,False,eo_only_partial


,comparison,llm_rows,template_rows,template_loaded,mean_overlap_at_5,mean_direction_accuracy,mean_action_consistency,llm_avg_word_count,note
0,current_loaded_conditions,200,0,False,1.0,1.0,1.0,95.0,Template-vs-LLM comparison remains partial bec...


In [359]:
# ============================================================
# Load template narratives and build template-vs-LLM summaries
# ============================================================

import pandas as pd
import numpy as np

# Try to load template narratives if not already loaded
if "template_df" not in globals() or template_df is None or not isinstance(template_df, pd.DataFrame) or template_df.empty:
    if "OPS_TRIAGE_TEMPLATE_PATH" in globals() and OPS_TRIAGE_TEMPLATE_PATH.exists():
        template_rows = read_jsonl(OPS_TRIAGE_TEMPLATE_PATH)
        template_df = pd.DataFrame(template_rows)
        print(f"Loaded template_df: {template_df.shape}")
    else:
        template_df = pd.DataFrame()
        print("[warn] Template narrative file unavailable; template_df remains empty")

def _safe_num_mean(s):
    s = pd.to_numeric(s, errors="coerce")
    return float(s.mean()) if len(s.dropna()) else np.nan

def _word_count(x):
    if pd.isna(x):
        return np.nan
    return len(str(x).split())

def _extract_text_col(df):
    for c in ["narrative", "summary", "text", "response_text", "final_narrative", "content"]:
        if c in df.columns:
            return c
    return None

def _extract_id_col(df):
    for c in ["eo_event_id", "event_id"]:
        if c in df.columns:
            return c
    return None

# ------------------------------------------------------------
# Build template merge if possible
# ------------------------------------------------------------
eo_template_df = pd.DataFrame()

if not template_df.empty and "eo_df" in globals() and isinstance(eo_df, pd.DataFrame) and not eo_df.empty:
    template_id_col = _extract_id_col(template_df)
    eo_id_col = "event_id" if "event_id" in eo_df.columns else None

    if template_id_col and eo_id_col:
        eo_template_df = eo_df.merge(
            template_df,
            left_on=eo_id_col,
            right_on=template_id_col,
            how="inner"
        )
        print(f"Built eo_template_df: {eo_template_df.shape}")
    else:
        print("[warn] Could not identify ID columns for EO/template merge")

# ------------------------------------------------------------
# Readability summary: LLM vs Template
# ------------------------------------------------------------
readability_rows = []

if "llm_df" in globals() and isinstance(llm_df, pd.DataFrame) and not llm_df.empty:
    llm_text_col = _extract_text_col(llm_df)
    if llm_text_col:
        readability_rows.append({
            "condition": "llm",
            "n_rows": len(llm_df),
            "avg_word_count": _safe_num_mean(llm_df[llm_text_col].apply(_word_count)),
            "text_column_used": llm_text_col,
        })

if not template_df.empty:
    template_text_col = _extract_text_col(template_df)
    if template_text_col:
        readability_rows.append({
            "condition": "template",
            "n_rows": len(template_df),
            "avg_word_count": _safe_num_mean(template_df[template_text_col].apply(_word_count)),
            "text_column_used": template_text_col,
        })

readability_summary_df = pd.DataFrame(readability_rows)

# ------------------------------------------------------------
# Template faithfulness proxies
# Note: true first-class template metrics require precomputed fields
# analogous to eo_llm_df. This provides best-effort comparability.
# ------------------------------------------------------------
template_faithfulness_summary_df = pd.DataFrame()

if not eo_template_df.empty:
    text_col = _extract_text_col(eo_template_df)

    if text_col and "top_drivers" in eo_template_df.columns:
        def _top_driver_mention_rate(row):
            txt = str(row[text_col]).lower() if pd.notna(row[text_col]) else ""
            drivers = row["top_drivers"] if isinstance(row["top_drivers"], list) else []
            names = []
            for d in drivers:
                if isinstance(d, dict) and "name" in d:
                    names.append(str(d["name"]).lower())
            if not names:
                return np.nan
            return float(any(name in txt for name in names))

        if "recommended_action_class" in eo_template_df.columns:
            def _action_match_proxy(row):
                txt = str(row[text_col]).lower() if pd.notna(row[text_col]) else ""
                action = str(row["recommended_action_class"]).lower()
                return float(action in txt)
        else:
            def _action_match_proxy(row):
                return np.nan

        template_faithfulness_summary_df = pd.DataFrame([
            {
                "condition": "template",
                "n_rows": len(eo_template_df),
                "top_driver_mention_rate_proxy": _safe_num_mean(eo_template_df.apply(_top_driver_mention_rate, axis=1)),
                "action_match_proxy": _safe_num_mean(eo_template_df.apply(_action_match_proxy, axis=1)),
                "note": "Best-effort template proxy metrics from text matching; not directly equivalent to eo_llm_df evaluation fields."
            }
        ])

# ------------------------------------------------------------
# Update comparison summary with actual template availability
# ------------------------------------------------------------
llm_avg_wc = np.nan
tpl_avg_wc = np.nan

if not readability_summary_df.empty:
    if (readability_summary_df["condition"] == "llm").any():
        llm_avg_wc = float(readability_summary_df.loc[readability_summary_df["condition"] == "llm", "avg_word_count"].iloc[0])
    if (readability_summary_df["condition"] == "template").any():
        tpl_avg_wc = float(readability_summary_df.loc[readability_summary_df["condition"] == "template", "avg_word_count"].iloc[0])

compare_condition_summary_df = pd.DataFrame([
    {
        "comparison": "template_vs_llm",
        "llm_rows": len(llm_df) if "llm_df" in globals() else np.nan,
        "template_rows": len(template_df) if "template_df" in globals() else np.nan,
        "template_loaded": not template_df.empty,
        "eo_template_rows": len(eo_template_df),
        "llm_avg_word_count": llm_avg_wc,
        "template_avg_word_count": tpl_avg_wc,
        "llm_mean_overlap_at_5": float(faithfulness_summary_df["mean_overlap_at_5"].iloc[0]) if "faithfulness_summary_df" in globals() and not faithfulness_summary_df.empty else np.nan,
        "llm_mean_direction_accuracy": float(faithfulness_summary_df["mean_direction_accuracy"].iloc[0]) if "faithfulness_summary_df" in globals() and not faithfulness_summary_df.empty else np.nan,
        "llm_mean_action_consistency": float(faithfulness_summary_df["mean_action_consistency"].iloc[0]) if "faithfulness_summary_df" in globals() and not faithfulness_summary_df.empty else np.nan,
        "template_top_driver_mention_rate_proxy": float(template_faithfulness_summary_df["top_driver_mention_rate_proxy"].iloc[0]) if not template_faithfulness_summary_df.empty else np.nan,
        "template_action_match_proxy": float(template_faithfulness_summary_df["action_match_proxy"].iloc[0]) if not template_faithfulness_summary_df.empty else np.nan,
        "note": "LLM metrics are first-class evaluation fields; template metrics are proxy text-match measures unless template evaluation fields are separately computed."
    }
])

print("[done] Template loading and comparison summary built")
display(template_df.head() if not template_df.empty else template_df)
display(readability_summary_df)
display(template_faithfulness_summary_df)
display(compare_condition_summary_df)

Built eo_template_df: (200, 22)
[done] Template loading and comparison summary built


,eo_event_id,persona,engine,run_metadata,narrative,metrics
0,3488961,ops_triage,template,{'run_id': '99cb1e40-9fbb-4c59-b62d-fd276b5c45...,"{'persona': 'ops_triage', 'summary': 'Risk is ...","{'overlap_at_K': 1.0, 'direction_accuracy': Tr..."
1,3488968,ops_triage,template,{'run_id': '99cb1e40-9fbb-4c59-b62d-fd276b5c45...,"{'persona': 'ops_triage', 'summary': 'Risk is ...","{'overlap_at_K': 1.0, 'direction_accuracy': Tr..."
2,3488985,ops_triage,template,{'run_id': '99cb1e40-9fbb-4c59-b62d-fd276b5c45...,"{'persona': 'ops_triage', 'summary': 'Risk is ...","{'overlap_at_K': 1.0, 'direction_accuracy': Tr..."
3,3489002,ops_triage,template,{'run_id': '99cb1e40-9fbb-4c59-b62d-fd276b5c45...,"{'persona': 'ops_triage', 'summary': 'Risk is ...","{'overlap_at_K': 1.0, 'direction_accuracy': Tr..."
4,3489003,ops_triage,template,{'run_id': '99cb1e40-9fbb-4c59-b62d-fd276b5c45...,"{'persona': 'ops_triage', 'summary': 'Risk is ...","{'overlap_at_K': 1.0, 'direction_accuracy': Tr..."


,condition,n_rows,avg_word_count,text_column_used
0,llm,200,95.0,narrative
1,template,200,95.0,narrative


,condition,n_rows,top_driver_mention_rate_proxy,action_match_proxy,note
0,template,200,1.0,1.0,Best-effort template proxy metrics from text m...


,comparison,llm_rows,template_rows,template_loaded,eo_template_rows,llm_avg_word_count,template_avg_word_count,llm_mean_overlap_at_5,llm_mean_direction_accuracy,llm_mean_action_consistency,template_top_driver_mention_rate_proxy,template_action_match_proxy,note
0,template_vs_llm,200,200,True,200,95.0,95.0,1.0,1.0,1.0,1.0,1.0,LLM metrics are first-class evaluation fields;...


In [360]:
# ============================================================
# Build eo_template_eval_df with symmetric evaluation metrics
# ============================================================

import pandas as pd
import numpy as np
import re

def _extract_text_col(df):
    for c in ["narrative", "summary", "text", "response_text", "final_narrative", "content"]:
        if c in df.columns:
            return c
    return None

def _extract_id_col(df):
    for c in ["eo_event_id", "event_id"]:
        if c in df.columns:
            return c
    return None

def _driver_names(top_drivers):
    if not isinstance(top_drivers, list):
        return []
    out = []
    for d in top_drivers:
        if isinstance(d, dict) and "name" in d:
            out.append(str(d["name"]))
    return out

def _driver_directions(top_drivers):
    if not isinstance(top_drivers, list):
        return {}
    out = {}
    for d in top_drivers:
        if isinstance(d, dict) and "name" in d:
            out[str(d["name"])] = str(d.get("direction", "")).strip()
    return out

def _mentioned_drivers(text, driver_names):
    if pd.isna(text):
        return []
    txt = str(text).lower()
    hits = []
    for name in driver_names:
        if str(name).lower() in txt:
            hits.append(name)
    return hits

def _direction_accuracy_proxy(text, top_drivers):
    if pd.isna(text) or not isinstance(top_drivers, list) or not top_drivers:
        return np.nan
    txt = str(text).lower()
    checks = []
    for d in top_drivers:
        if not isinstance(d, dict) or "name" not in d:
            continue
        name = str(d["name"])
        direction = str(d.get("direction", "")).strip()
        if name.lower() not in txt:
            continue

        local_window = txt
        pos_tokens = ["increase", "increases", "higher", "elevated", "raises", "up"]
        neg_tokens = ["decrease", "decreases", "lower", "reduces", "down", "mitigat"]

        has_pos = any(tok in local_window for tok in pos_tokens)
        has_neg = any(tok in local_window for tok in neg_tokens)

        if direction == "+":
            checks.append(float(has_pos and not has_neg) if (has_pos or has_neg) else np.nan)
        elif direction == "-":
            checks.append(float(has_neg and not has_pos) if (has_pos or has_neg) else np.nan)

    checks = pd.Series(checks, dtype="float")
    return float(checks.mean()) if len(checks.dropna()) else np.nan

def _action_consistency_proxy(text, action):
    if pd.isna(text) or pd.isna(action):
        return np.nan
    txt = str(text).lower()
    action = str(action).lower()
    return float(action in txt)

def _evidence_disclosure_proxy(text, thin_file_flag, evidence_strength, monitoring):
    if pd.isna(text):
        return np.nan
    txt = str(text).lower()

    needs_disclosure = False
    if pd.notna(thin_file_flag) and bool(thin_file_flag):
        needs_disclosure = True
    if pd.notna(evidence_strength) and str(evidence_strength).upper() == "LOW":
        needs_disclosure = True
    if isinstance(monitoring, dict) and str(monitoring.get("drift_status", "")).upper() == "WARN":
        needs_disclosure = True

    disclosure_terms = [
        "limited evidence", "low coverage", "thin-file", "thin file",
        "uncertain", "uncertainty", "drift", "warn", "caution", "reduced confidence"
    ]
    has_disclosure = any(term in txt for term in disclosure_terms)

    if not needs_disclosure:
        return np.nan
    return float(has_disclosure)

# ------------------------------------------------------------
# Load template_df if needed
# ------------------------------------------------------------
if "template_df" not in globals() or template_df is None or not isinstance(template_df, pd.DataFrame):
    template_df = pd.DataFrame()

if template_df.empty:
    if "OPS_TRIAGE_TEMPLATE_PATH" in globals() and OPS_TRIAGE_TEMPLATE_PATH.exists():
        template_df = pd.DataFrame(read_jsonl(OPS_TRIAGE_TEMPLATE_PATH))
        print(f"Loaded template_df: {template_df.shape}")

# ------------------------------------------------------------
# Build merged EO/template frame
# ------------------------------------------------------------
eo_template_eval_df = pd.DataFrame()

if "eo_df" in globals() and isinstance(eo_df, pd.DataFrame) and not eo_df.empty and not template_df.empty:
    eo_id_col = "event_id" if "event_id" in eo_df.columns else None
    template_id_col = _extract_id_col(template_df)
    template_text_col = _extract_text_col(template_df)

    if eo_id_col and template_id_col and template_text_col:
        eo_template_eval_df = eo_df.merge(
            template_df,
            left_on=eo_id_col,
            right_on=template_id_col,
            how="inner"
        )

        eo_template_eval_df["mentioned_drivers"] = eo_template_eval_df.apply(
            lambda row: _mentioned_drivers(row[template_text_col], _driver_names(row["top_drivers"])),
            axis=1
        )

        eo_template_eval_df["overlap_at_5"] = eo_template_eval_df.apply(
            lambda row: (
                len(set(row["mentioned_drivers"]) & set(_driver_names(row["top_drivers"]))) /
                max(len(_driver_names(row["top_drivers"])), 1)
            ),
            axis=1
        )

        eo_template_eval_df["direction_accuracy"] = eo_template_eval_df.apply(
            lambda row: _direction_accuracy_proxy(row[template_text_col], row["top_drivers"]),
            axis=1
        )

        eo_template_eval_df["action_consistency"] = eo_template_eval_df.apply(
            lambda row: _action_consistency_proxy(row[template_text_col], row["recommended_action_class"]),
            axis=1
        )

        eo_template_eval_df["has_evidence_disclosure"] = eo_template_eval_df.apply(
            lambda row: _evidence_disclosure_proxy(
                row[template_text_col],
                row.get("thin_file_flag"),
                row.get("evidence_strength"),
                row.get("monitoring"),
            ),
            axis=1
        )

        print(f"Built eo_template_eval_df: {eo_template_eval_df.shape}")
    else:
        print("[warn] Could not identify required columns to build eo_template_eval_df")

# ------------------------------------------------------------
# First-class template summary
# ------------------------------------------------------------
template_eval_summary_df = pd.DataFrame()

if not eo_template_eval_df.empty:
    template_eval_summary_df = pd.DataFrame([
        {
            "condition": "template",
            "n_rows": len(eo_template_eval_df),
            "mean_overlap_at_5": float(pd.to_numeric(eo_template_eval_df["overlap_at_5"], errors="coerce").mean()),
            "mean_direction_accuracy": float(pd.to_numeric(eo_template_eval_df["direction_accuracy"], errors="coerce").mean()),
            "mean_action_consistency": float(pd.to_numeric(eo_template_eval_df["action_consistency"], errors="coerce").mean()),
            "mean_has_evidence_disclosure": float(pd.to_numeric(eo_template_eval_df["has_evidence_disclosure"], errors="coerce").mean()),
        }
    ])

# ------------------------------------------------------------
# Upgrade comparison summary to symmetric evaluation metrics
# ------------------------------------------------------------
llm_overlap = np.nan
llm_dir = np.nan
llm_action = np.nan
llm_disc = np.nan

if "faithfulness_summary_df" in globals() and not faithfulness_summary_df.empty:
    if "mean_overlap_at_5" in faithfulness_summary_df.columns:
        llm_overlap = float(faithfulness_summary_df["mean_overlap_at_5"].iloc[0])
    elif "driver_overlap_at_k" in faithfulness_summary_df.columns:
        llm_overlap = float(faithfulness_summary_df["driver_overlap_at_k"].iloc[0])

    if "mean_direction_accuracy" in faithfulness_summary_df.columns:
        llm_dir = float(faithfulness_summary_df["mean_direction_accuracy"].iloc[0])

    if "mean_action_consistency" in faithfulness_summary_df.columns:
        llm_action = float(faithfulness_summary_df["mean_action_consistency"].iloc[0])
    elif "action_consistency_rate" in faithfulness_summary_df.columns:
        llm_action = float(faithfulness_summary_df["action_consistency_rate"].iloc[0])

    if "mean_has_evidence_disclosure" in faithfulness_summary_df.columns:
        llm_disc = float(faithfulness_summary_df["mean_has_evidence_disclosure"].iloc[0])

template_overlap = float(template_eval_summary_df["mean_overlap_at_5"].iloc[0]) if not template_eval_summary_df.empty else np.nan
template_dir = float(template_eval_summary_df["mean_direction_accuracy"].iloc[0]) if not template_eval_summary_df.empty else np.nan
template_action = float(template_eval_summary_df["mean_action_consistency"].iloc[0]) if not template_eval_summary_df.empty else np.nan
template_disc = float(template_eval_summary_df["mean_has_evidence_disclosure"].iloc[0]) if not template_eval_summary_df.empty else np.nan

# preserve readability figures if already computed
llm_avg_wc = np.nan
template_avg_wc = np.nan
if "readability_summary_df" in globals() and not readability_summary_df.empty and "condition" in readability_summary_df.columns:
    if (readability_summary_df["condition"] == "llm").any():
        llm_avg_wc = float(readability_summary_df.loc[readability_summary_df["condition"] == "llm", "avg_word_count"].iloc[0])
    if (readability_summary_df["condition"] == "template").any():
        template_avg_wc = float(readability_summary_df.loc[readability_summary_df["condition"] == "template", "avg_word_count"].iloc[0])

compare_condition_summary_df = pd.DataFrame([
    {
        "comparison": "template_vs_llm_first_class",
        "llm_rows": len(eo_llm_df) if "eo_llm_df" in globals() else np.nan,
        "template_rows": len(eo_template_eval_df),
        "llm_mean_overlap_at_5": llm_overlap,
        "template_mean_overlap_at_5": template_overlap,
        "llm_mean_direction_accuracy": llm_dir,
        "template_mean_direction_accuracy": template_dir,
        "llm_mean_action_consistency": llm_action,
        "template_mean_action_consistency": template_action,
        "llm_mean_has_evidence_disclosure": llm_disc,
        "template_mean_has_evidence_disclosure": template_disc,
        "llm_avg_word_count": llm_avg_wc,
        "template_avg_word_count": template_avg_wc,
        "note": "Template metrics are now computed on the EO-linked template narratives using the same evaluation dimensions as the LLM condition, though direction/action/disclosure remain text-derived proxies."
    }
])

print("[done] Built eo_template_eval_df and upgraded comparison summary")
display(template_eval_summary_df)
display(compare_condition_summary_df)

Built eo_template_eval_df: (200, 27)
[done] Built eo_template_eval_df and upgraded comparison summary


,condition,n_rows,mean_overlap_at_5,mean_direction_accuracy,mean_action_consistency,mean_has_evidence_disclosure
0,template,200,1.0,0.05,1.0,1.0


,comparison,llm_rows,template_rows,llm_mean_overlap_at_5,template_mean_overlap_at_5,llm_mean_direction_accuracy,template_mean_direction_accuracy,llm_mean_action_consistency,template_mean_action_consistency,llm_mean_has_evidence_disclosure,template_mean_has_evidence_disclosure,llm_avg_word_count,template_avg_word_count,note
0,template_vs_llm_first_class,200,200,1.0,1.0,1.0,0.05,1.0,1.0,1.0,1.0,95.0,95.0,Template metrics are now computed on the EO-li...


In [361]:
# ============================================================
# Thin-file vs non-thin-file delta table (robust diagnostic) RQ5 / H4
# ============================================================

import pandas as pd
import numpy as np
import re

def _safe_mean(series):
    s = pd.to_numeric(series, errors="coerce")
    return float(s.mean()) if len(s.dropna()) else np.nan

def _safe_rate_bool(series):
    if series is None or len(series) == 0:
        return np.nan
    s = pd.Series(series)
    if s.dtype == bool:
        return float(s.mean())
    s = s.astype(str).str.strip().str.lower()
    mapped = s.map({
        "true": 1.0, "false": 0.0,
        "1": 1.0, "0": 0.0,
        "yes": 1.0, "no": 0.0
    })
    return float(mapped.mean()) if len(mapped.dropna()) else np.nan

def _extract_text_col(df):
    for c in ["narrative", "summary", "text", "response_text", "final_narrative", "content"]:
        if c in df.columns:
            return c
    return None

def _extract_monitoring_drift_status(x):
    if isinstance(x, dict):
        return x.get("drift_status")
    return None

def _extract_action_from_text(text):
    if pd.isna(text):
        return None
    txt = str(text).lower()
    patterns = [
        ("step-up", [r"\bstep[- ]?up\b", r"\badditional verification\b", r"\botp\b", r"\bkyc\b"]),
        ("block", [r"\bblock\b", r"\bdecline\b", r"\breject\b"]),
        ("review", [r"\breview\b", r"\bmanual review\b", r"\binvestigat"]),
        ("approve", [r"\bapprove\b", r"\ballow\b", r"\baccept\b"]),
        ("monitor", [r"\bmonitor\b", r"\bwatch\b"]),
    ]
    for label, pats in patterns:
        if any(re.search(p, txt) for p in pats):
            return label
    return None

def _needs_disclosure(thin_file_flag, evidence_strength, monitoring):
    if pd.notna(thin_file_flag) and bool(thin_file_flag):
        return True
    if pd.notna(evidence_strength) and str(evidence_strength).upper() == "LOW":
        return True
    if isinstance(monitoring, dict) and str(monitoring.get("drift_status", "")).upper() == "WARN":
        return True
    return False

def _has_disclosure_text(text):
    if pd.isna(text):
        return False
    txt = str(text).lower()
    terms = [
        "limited evidence", "low coverage", "thin-file", "thin file",
        "uncertain", "uncertainty", "drift", "warn", "caution",
        "reduced confidence", "limited supporting signals"
    ]
    return any(term in txt for term in terms)

thinfile_delta_df = pd.DataFrame()
thinfile_delta_summary_df = pd.DataFrame()

if "eo_llm_df" not in globals() or not isinstance(eo_llm_df, pd.DataFrame) or eo_llm_df.empty:
    thinfile_delta_summary_df = pd.DataFrame([{
        "comparison": "thin_file_minus_non_thin_file",
        "status": "eo_llm_df_unavailable",
        "note": "Merged EO/LLM dataframe is unavailable or empty."
    }])
else:
    df = eo_llm_df.copy()

    if "thin_file_flag" not in df.columns:
        thinfile_delta_summary_df = pd.DataFrame([{
            "comparison": "thin_file_minus_non_thin_file",
            "status": "thin_file_flag_missing",
            "note": "eo_llm_df does not contain thin_file_flag."
        }])
    else:
        text_col = _extract_text_col(df)

        if text_col is None:
            thinfile_delta_summary_df = pd.DataFrame([{
                "comparison": "thin_file_minus_non_thin_file",
                "status": "narrative_text_missing",
                "note": "Could not identify a narrative text column in eo_llm_df."
            }])
        else:
            df["thin_file_group"] = df["thin_file_flag"].fillna(False).astype(bool).map({
                True: "thin_file",
                False: "non_thin_file"
            })

            if "monitoring" in df.columns:
                df["drift_status"] = df["monitoring"].apply(_extract_monitoring_drift_status)
            else:
                df["drift_status"] = None

            df["narrative_action_proxy"] = df[text_col].apply(_extract_action_from_text)
            df["is_step_up_proxy"] = df["narrative_action_proxy"].eq("step-up")
            df["needs_disclosure"] = df.apply(
                lambda row: _needs_disclosure(
                    row.get("thin_file_flag"),
                    row.get("evidence_strength"),
                    row.get("monitoring")
                ),
                axis=1
            )
            df["has_disclosure_text_proxy"] = df[text_col].apply(_has_disclosure_text)

            df["action_consistency_metric"] = pd.to_numeric(df["action_consistency"], errors="coerce") if "action_consistency" in df.columns else np.nan
            df["overlap_at_5_metric"] = pd.to_numeric(df["overlap_at_5"], errors="coerce") if "overlap_at_5" in df.columns else np.nan
            df["direction_accuracy_metric"] = pd.to_numeric(df["direction_accuracy"], errors="coerce") if "direction_accuracy" in df.columns else np.nan
            df["has_evidence_disclosure_metric"] = pd.to_numeric(df["has_evidence_disclosure"], errors="coerce") if "has_evidence_disclosure" in df.columns else np.nan

            group_rows = []
            for grp, sdf in df.groupby("thin_file_group", dropna=False):
                group_rows.append({
                    "group": grp,
                    "n_rows": len(sdf),
                    "share_of_eval_set": len(sdf) / len(df),
                    "mean_overlap_at_5": _safe_mean(sdf["overlap_at_5_metric"]),
                    "mean_direction_accuracy": _safe_mean(sdf["direction_accuracy_metric"]),
                    "mean_action_consistency": _safe_mean(sdf["action_consistency_metric"]),
                    "mean_has_evidence_disclosure_metric": _safe_mean(sdf["has_evidence_disclosure_metric"]),
                    "disclosure_text_proxy_rate": _safe_rate_bool(sdf["has_disclosure_text_proxy"]),
                    "step_up_proxy_rate": _safe_rate_bool(sdf["is_step_up_proxy"]),
                    "mean_score": _safe_mean(sdf["score"]) if "score" in sdf.columns else np.nan,
                    "mean_data_coverage_score": _safe_mean(sdf["data_coverage_score"]) if "data_coverage_score" in sdf.columns else np.nan,
                    "low_evidence_rate": _safe_rate_bool(sdf["evidence_strength"].astype(str).str.upper().eq("LOW")) if "evidence_strength" in sdf.columns else np.nan,
                    "warn_drift_rate": _safe_rate_bool(sdf["drift_status"].astype(str).str.upper().eq("WARN")),
                })

            thinfile_delta_df = pd.DataFrame(group_rows)
            thinfile_summary_df = thinfile_delta_df.copy()

            thin_row = thinfile_delta_df.loc[thinfile_delta_df["group"] == "thin_file"]
            non_row = thinfile_delta_df.loc[thinfile_delta_df["group"] == "non_thin_file"]

            if len(thin_row) and len(non_row):
                thin_row = thin_row.iloc[0]
                non_row = non_row.iloc[0]

                def _delta(col):
                    a = thin_row.get(col, np.nan)
                    b = non_row.get(col, np.nan)
                    try:
                        if pd.isna(a) or pd.isna(b):
                            return np.nan
                        return float(a - b)
                    except Exception:
                        return np.nan

                thinfile_delta_summary_df = pd.DataFrame([{
                    "comparison": "thin_file_minus_non_thin_file",
                    "status": "ok",
                    "thin_file_n": float(thin_row["n_rows"]),
                    "non_thin_file_n": float(non_row["n_rows"]),
                    "delta_share_of_eval_set": _delta("share_of_eval_set"),
                    "delta_mean_overlap_at_5": _delta("mean_overlap_at_5"),
                    "delta_mean_direction_accuracy": _delta("mean_direction_accuracy"),
                    "delta_mean_action_consistency": _delta("mean_action_consistency"),
                    "delta_mean_has_evidence_disclosure_metric": _delta("mean_has_evidence_disclosure_metric"),
                    "delta_disclosure_text_proxy_rate": _delta("disclosure_text_proxy_rate"),
                    "delta_step_up_proxy_rate": _delta("step_up_proxy_rate"),
                    "delta_mean_score": _delta("mean_score"),
                    "delta_mean_data_coverage_score": _delta("mean_data_coverage_score"),
                    "delta_low_evidence_rate": _delta("low_evidence_rate"),
                    "delta_warn_drift_rate": _delta("warn_drift_rate"),
                    "note": "Positive values mean the metric is higher for thin-file cases."
                }])
            else:
                counts = df["thin_file_group"].value_counts(dropna=False).to_dict()
                thinfile_delta_summary_df = pd.DataFrame([{
                    "comparison": "thin_file_minus_non_thin_file",
                    "status": "one_group_only",
                    "thin_file_n": counts.get("thin_file", 0),
                    "non_thin_file_n": counts.get("non_thin_file", 0),
                    "note": "Delta unavailable because only one thin_file_group is present in eo_llm_df."
                }])

print("[done] Built robust thin-file delta outputs")
display(thinfile_delta_df)
display(thinfile_delta_summary_df)

[done] Built robust thin-file delta outputs


,group,n_rows,share_of_eval_set,mean_overlap_at_5,mean_direction_accuracy,mean_action_consistency,mean_has_evidence_disclosure_metric,disclosure_text_proxy_rate,step_up_proxy_rate,mean_score,mean_data_coverage_score,low_evidence_rate,warn_drift_rate
0,thin_file,200,1.0,1.0,1.0,1.0,1.0,1.0,0.0,0.014225,0.34,1.0,0.0


,comparison,status,thin_file_n,non_thin_file_n,note
0,thin_file_minus_non_thin_file,one_group_only,200,0,Delta unavailable because only one thin_file_g...


In [362]:
# ============================================================
# Thesis-ready interpretation cell
# Proposal-aligned synthesis of current notebook results
# ============================================================

import math
import pandas as pd
import numpy as np

def _fmt(x, digits=3):
    try:
        if x is None or (isinstance(x, float) and (math.isnan(x) or math.isinf(x))):
            return "NA"
        return f"{float(x):.{digits}f}"
    except Exception:
        return str(x)

def _get_first(df, col):
    if isinstance(df, pd.DataFrame) and not df.empty and col in df.columns:
        return df[col].iloc[0]
    return np.nan

llm_overlap = _get_first(faithfulness_summary_df, "mean_overlap_at_5")
llm_dir = _get_first(faithfulness_summary_df, "mean_direction_accuracy")
llm_action = _get_first(faithfulness_summary_df, "mean_action_consistency")
llm_disc = _get_first(faithfulness_summary_df, "mean_has_evidence_disclosure")

tpl_overlap = _get_first(compare_condition_summary_df, "template_mean_overlap_at_5")
tpl_dir = _get_first(compare_condition_summary_df, "template_mean_direction_accuracy")
tpl_action = _get_first(compare_condition_summary_df, "template_mean_action_consistency")
tpl_disc = _get_first(compare_condition_summary_df, "template_mean_has_evidence_disclosure")

llm_wc = _get_first(compare_condition_summary_df, "llm_avg_word_count")
tpl_wc = _get_first(compare_condition_summary_df, "template_avg_word_count")

has_template = bool(_get_first(compare_condition_summary_df, "template_rows")) if "compare_condition_summary_df" in globals() and not compare_condition_summary_df.empty else False
has_perturb_eo = bool(len(perturbed_eo_df) > 0) if "perturbed_eo_df" in globals() and isinstance(perturbed_eo_df, pd.DataFrame) else False
has_perturb_llm = bool(len(perturbed_llm_df) > 0) if "perturbed_llm_df" in globals() and isinstance(perturbed_llm_df, pd.DataFrame) else False

print("# Results synthesis\n")

print("## RQ1 — Faithfulness")
print(
    f"- Constrained LLM narratives achieved mean overlap@5={_fmt(llm_overlap)}, "
    f"direction accuracy={_fmt(llm_dir)}, and action consistency={_fmt(llm_action)} "
    f"on the EO-linked evaluation subset."
)
if has_template:
    print(
        f"- Template narratives on the same EO-linked setup achieved overlap@5={_fmt(tpl_overlap)}, "
        f"direction accuracy={_fmt(tpl_dir)}, and action consistency={_fmt(tpl_action)}."
    )
    print(
        "- Interpretation: the notebook now supports a direct template-vs-LLM comparison on the core "
        "faithfulness dimensions, though the template-side metrics remain text-derived proxies rather than "
        "precomputed validator outputs."
    )
else:
    print("- Template comparison is not yet available in the current run state.")
print()

print("## RQ2 — Stability / perturbation")
if has_perturb_eo and has_perturb_llm:
    print("- Both perturbed EO and perturbed LLM outputs are present, so full perturbation-response analysis is available.")
elif has_perturb_eo and not has_perturb_llm:
    print(
        "- Perturbed EO artifacts are present, but perturbed LLM narratives are absent. "
        "Therefore the current notebook supports only partial stability analysis at the EO/action-change level, "
        "not full narrative-response stability."
    )
else:
    print("- Perturbation artifacts are not available, so RQ2 remains not yet evaluated in this run.")
print()

print("## RQ3 — Decision utility")
print(
    "- No human simulatability study is currently implemented in this notebook run. "
    "Decision utility is therefore evidenced indirectly through the proxy suite "
    "(faithfulness, disclosure behavior, action consistency, and governance summaries)."
)
print()

print("## RQ4 — Drift awareness")
if "drift_summary_df" in globals() and isinstance(drift_summary_df, pd.DataFrame) and not drift_summary_df.empty:
    print("- Drift-aware summary table is populated.")
    print(
        "- Interpretation: the notebook now captures drift-stratified evaluation from the EO.monitoring field, "
        "which is aligned to the proposal’s governance objective, though conclusions depend on the observed drift-status mix."
    )
else:
    print("- Drift summary is unavailable in this run.")
print()

print("## RQ5 — Thin-file robustness")
if "thinfile_summary_df" in globals() and isinstance(thinfile_summary_df, pd.DataFrame) and not thinfile_summary_df.empty:
    print("- Thin-file summary table is populated, including faithfulness/disclosure behavior stratified by thin_file_flag.")
    print(
        "- Interpretation: this directly supports the proposal’s thin-file governance objective by checking whether "
        "low-evidence cases are treated differently in disclosure and action behavior."
    )
else:
    print("- Thin-file summary is unavailable in this run.")
print()

print("## H1 — Readability vs faithfulness")
if has_template:
    print(
        f"- Average word count: LLM={_fmt(llm_wc)}, template={_fmt(tpl_wc)}. "
        "This provides a first readability proxy for comparing the two conditions."
    )
    print(
        "- Provisional interpretation: H1 can be discussed, but should be framed cautiously because readability is "
        "currently proxied mainly by length/verbosity rather than a richer readability metric or human judgment."
    )
else:
    print("- H1 remains partial because the template comparison is incomplete.")
print()

print("## H2 — Value of constraints")
print(
    "- The current notebook structure supports constrained-output analysis and validator summaries, "
    "but unconstrained ablation results are not yet clearly present in the current exported result pack. "
    "Therefore H2 remains partial/incomplete in this run."
)
print()

print("## H3 — Drift messaging")
print(
    "- This is partially supported by the drift summary and evidence-disclosure summaries. "
    "A stronger claim would require explicit analysis of whether WARN cases shift action recommendations "
    "toward step-up behavior."
)
print()

print("## H4 — Thin-file governance")
print(
    "- This is partially supported by thin-file stratified summaries. "
    "A stronger final result would compare thin-file vs non-thin-file action behavior and disclosure rates explicitly."
)
print()

print("## Overall status")
print(
    "- The notebook now delivers a stable, proposal-aligned automated evaluation pack covering core faithfulness, "
    "thin-file governance, drift-aware summaries, validator metadata, and partial perturbation analysis."
)
print(
    "- The strongest remaining gaps for a first-class final thesis result are: "
    "(1) richer template-vs-LLM comparability, "
    "(2) explicit unconstrained ablation results for H2, and "
    "(3) full perturbation-response narrative analysis if perturbed LLM outputs are generated."
)

# Results synthesis

## RQ1 — Faithfulness
- Constrained LLM narratives achieved mean overlap@5=1.000, direction accuracy=1.000, and action consistency=1.000 on the EO-linked evaluation subset.
- Template narratives on the same EO-linked setup achieved overlap@5=1.000, direction accuracy=0.050, and action consistency=1.000.
- Interpretation: the notebook now supports a direct template-vs-LLM comparison on the core faithfulness dimensions, though the template-side metrics remain text-derived proxies rather than precomputed validator outputs.

## RQ2 — Stability / perturbation
- Perturbed EO artifacts are present, but perturbed LLM narratives are absent. Therefore the current notebook supports only partial stability analysis at the EO/action-change level, not full narrative-response stability.

## RQ3 — Decision utility
- No human simulatability study is currently implemented in this notebook run. Decision utility is therefore evidenced indirectly through the proxy suite (faithfulness, d

In [363]:
# ============================================================
# Proposal-grade claims table
# Labels each RQ / H as Supported / Partial / Not yet tested
# with evidence pointers to notebook-generated tables
# ============================================================

import pandas as pd
import numpy as np

def _first(df, col, default=np.nan):
    if isinstance(df, pd.DataFrame) and not df.empty and col in df.columns:
        return df[col].iloc[0]
    return default

def _has_rows(df):
    return isinstance(df, pd.DataFrame) and not df.empty

def _fmt_metric(x, digits=3):
    try:
        if pd.isna(x):
            return "NA"
        return f"{float(x):.{digits}f}"
    except Exception:
        return str(x)

claims_rows = []

# ------------------------------------------------------------
# Gather current evidence state from summary tables
# ------------------------------------------------------------
llm_overlap = _first(globals().get("faithfulness_summary_df"), "mean_overlap_at_5")
llm_dir = _first(globals().get("faithfulness_summary_df"), "mean_direction_accuracy")
llm_action = _first(globals().get("faithfulness_summary_df"), "mean_action_consistency")
llm_disc = _first(globals().get("faithfulness_summary_df"), "mean_has_evidence_disclosure")

template_loaded = bool(_first(globals().get("compare_condition_summary_df"), "template_rows", 0))
template_overlap = _first(globals().get("compare_condition_summary_df"), "template_mean_overlap_at_5")
template_dir = _first(globals().get("compare_condition_summary_df"), "template_mean_direction_accuracy")
template_action = _first(globals().get("compare_condition_summary_df"), "template_mean_action_consistency")
template_disc = _first(globals().get("compare_condition_summary_df"), "template_mean_has_evidence_disclosure")
llm_wc = _first(globals().get("compare_condition_summary_df"), "llm_avg_word_count")
template_wc = _first(globals().get("compare_condition_summary_df"), "template_avg_word_count")

has_drift = _has_rows(globals().get("drift_summary_df"))
has_thinfile = _has_rows(globals().get("thinfile_summary_df"))
has_validator = _has_rows(globals().get("validator_summary_df"))
has_validator_failures = _has_rows(globals().get("validator_failure_summary_df"))

has_perturb_response = _has_rows(globals().get("perturbation_response_summary_df"))
perturb_status = _first(globals().get("perturbation_response_summary_df"), "status", None)

has_thinfile_delta = _has_rows(globals().get("thinfile_delta_summary_df"))
thinfile_delta_status = _first(globals().get("thinfile_delta_summary_df"), "status", None)

thinfile_only_summary = _has_rows(globals().get("thinfile_thesis_summary_df"))

# ------------------------------------------------------------
# RQ1 Faithfulness
# ------------------------------------------------------------
if _has_rows(globals().get("faithfulness_summary_df")):
    status = "Supported"
    rationale = (
        f"Faithfulness table populated with overlap@5={_fmt_metric(llm_overlap)}, "
        f"direction accuracy={_fmt_metric(llm_dir)}, and "
        f"action consistency={_fmt_metric(llm_action)}."
    )
    evidence = "faithfulness_summary.csv"
else:
    status = "Not yet tested"
    rationale = "Faithfulness summary table is unavailable."
    evidence = ""
claims_rows.append({
    "claim_id": "RQ1",
    "claim_type": "research_question",
    "theme": "faithfulness",
    "status": status,
    "rationale": rationale,
    "evidence_tables": evidence,
})

# ------------------------------------------------------------
# RQ2 Stability
# ------------------------------------------------------------
if has_perturb_response:
    if str(perturb_status) == "eo_only_partial":
        status = "Partial"
        rationale = (
            "Perturbation summary is available only at the EO level because perturbed LLM narratives are missing; "
            "full narrative stability analysis is not yet possible."
        )
    else:
        status = "Supported"
        rationale = "Perturbation response summary is populated."
    evidence = "perturbation_response_summary.csv; perturbation_build_summary.csv"
else:
    status = "Not yet tested"
    rationale = "No perturbation response summary is available."
    evidence = ""
claims_rows.append({
    "claim_id": "RQ2",
    "claim_type": "research_question",
    "theme": "stability",
    "status": status,
    "rationale": rationale,
    "evidence_tables": evidence,
})

# ------------------------------------------------------------
# RQ3 Decision utility
# ------------------------------------------------------------
status = "Partial"
rationale = (
    "No human simulatability study is present in the current notebook run. "
    "Decision utility is only indirectly evidenced through proxy metrics such as faithfulness, "
    "disclosure behavior, and action consistency."
)
evidence = "faithfulness_summary.csv; thinfile_summary.csv; drift_summary.csv"
claims_rows.append({
    "claim_id": "RQ3",
    "claim_type": "research_question",
    "theme": "decision_utility",
    "status": status,
    "rationale": rationale,
    "evidence_tables": evidence,
})

# ------------------------------------------------------------
# RQ4 Drift awareness
# ------------------------------------------------------------
if has_drift:
    status = "Partial"
    rationale = (
        "Drift summary is populated from EO.monitoring, so drift-stratified analysis is available. "
        "However, stronger support would require explicit evidence that WARN cases change action calibration "
        "or disclosure behavior in the intended direction."
    )
    evidence = "drift_summary.csv"
else:
    status = "Not yet tested"
    rationale = "Drift summary is unavailable."
    evidence = ""
claims_rows.append({
    "claim_id": "RQ4",
    "claim_type": "research_question",
    "theme": "drift_awareness",
    "status": status,
    "rationale": rationale,
    "evidence_tables": evidence,
})

# ------------------------------------------------------------
# RQ5 Thin-file robustness
# ------------------------------------------------------------
if thinfile_only_summary:
    status = "Partial"
    rationale = (
        "Thin-file-specific analysis is available, but the evaluated narrative subset contains only thin-file cases, "
        "so comparative thin-file vs non-thin-file deltas cannot be estimated from this sample."
    )
    evidence = "thinfile_summary.csv; thinfile_delta_summary.csv; thinfile_thesis_summary.csv"
elif has_thinfile and has_thinfile_delta and str(thinfile_delta_status) == "ok":
    status = "Supported"
    rationale = "Thin-file and non-thin-file comparative summaries are both available."
    evidence = "thinfile_summary.csv; thinfile_delta_summary.csv"
elif has_thinfile:
    status = "Partial"
    rationale = "Thin-file summary is available, but the comparative delta analysis is incomplete."
    evidence = "thinfile_summary.csv"
else:
    status = "Not yet tested"
    rationale = "Thin-file summary is unavailable."
    evidence = ""
claims_rows.append({
    "claim_id": "RQ5",
    "claim_type": "research_question",
    "theme": "thin_file_robustness",
    "status": status,
    "rationale": rationale,
    "evidence_tables": evidence,
})

# ------------------------------------------------------------
# RQ6 Portability
# ------------------------------------------------------------
claims_rows.append({
    "claim_id": "RQ6",
    "claim_type": "research_question",
    "theme": "portability_optional",
    "status": "Not yet tested",
    "rationale": "Optional portability slices are not present in the current notebook run.",
    "evidence_tables": "",
})

# ------------------------------------------------------------
# H1 Readability / clarity
# ------------------------------------------------------------
if template_loaded and _has_rows(globals().get("readability_summary_df")):
    status = "Partial"
    rationale = (
        f"Readability comparison is available with average word count "
        f"LLM={_fmt_metric(llm_wc)} vs template={_fmt_metric(template_wc)}. "
        "This supports a first comparison, but stronger evidence would require richer readability measures "
        "or human evaluation."
    )
    evidence = "readability_summary.csv; compare_condition_summary.csv"
else:
    status = "Not yet tested"
    rationale = "Template-vs-LLM readability comparison is incomplete."
    evidence = ""
claims_rows.append({
    "claim_id": "H1",
    "claim_type": "hypothesis",
    "theme": "readability_vs_faithfulness",
    "status": status,
    "rationale": rationale,
    "evidence_tables": evidence,
})

# ------------------------------------------------------------
# H2 Constraint value
# ------------------------------------------------------------
if has_validator:
    status = "Partial"
    rationale = (
        "Validator summaries are available, but unconstrained ablation results are not clearly present in the current result pack. "
        "This supports only partial assessment of the value of constraints."
    )
    evidence = "validator_summary.csv; validator_failure_summary.csv"
else:
    status = "Not yet tested"
    rationale = "Validator or unconstrained ablation evidence is unavailable."
    evidence = ""
claims_rows.append({
    "claim_id": "H2",
    "claim_type": "hypothesis",
    "theme": "constraint_value",
    "status": status,
    "rationale": rationale,
    "evidence_tables": evidence,
})

# ------------------------------------------------------------
# H3 Drift messaging
# ------------------------------------------------------------
if has_drift:
    status = "Partial"
    rationale = (
        "Drift-related summaries exist, but the current notebook does not yet demonstrate a strong causal link "
        "between drift disclosure and safer or more calibrated action behavior."
    )
    evidence = "drift_summary.csv; thinfile_summary.csv"
else:
    status = "Not yet tested"
    rationale = "Drift-aware evidence is unavailable."
    evidence = ""
claims_rows.append({
    "claim_id": "H3",
    "claim_type": "hypothesis",
    "theme": "drift_messaging",
    "status": status,
    "rationale": rationale,
    "evidence_tables": evidence,
})

# ------------------------------------------------------------
# H4 Thin-file governance
# ------------------------------------------------------------
if thinfile_only_summary:
    status = "Partial"
    rationale = (
        "Thin-file narrative behavior is directly analyzed, but because the evaluated sample contains only thin-file cases, "
        "the claim that thin-file messaging reduces aggressive actions relative to non-thin-file cases remains only partially tested."
    )
    evidence = "thinfile_summary.csv; thinfile_delta_summary.csv; thinfile_thesis_summary.csv"
elif has_thinfile and has_thinfile_delta and str(thinfile_delta_status) == "ok":
    status = "Supported"
    rationale = "Comparative thin-file vs non-thin-file governance summaries are available."
    evidence = "thinfile_summary.csv; thinfile_delta_summary.csv"
elif has_thinfile:
    status = "Partial"
    rationale = "Thin-file analysis is available, but comparative evidence is incomplete."
    evidence = "thinfile_summary.csv"
else:
    status = "Not yet tested"
    rationale = "Thin-file governance evidence is unavailable."
    evidence = ""
claims_rows.append({
    "claim_id": "H4",
    "claim_type": "hypothesis",
    "theme": "thin_file_governance",
    "status": status,
    "rationale": rationale,
    "evidence_tables": evidence,
})

# ------------------------------------------------------------
# Final table
# ------------------------------------------------------------
claims_table_df = pd.DataFrame(claims_rows)

status_order = pd.CategoricalDtype(
    categories=["Supported", "Partial", "Not yet tested"],
    ordered=True
)
claims_table_df["status"] = claims_table_df["status"].astype(status_order)
claims_table_df = claims_table_df.sort_values(["claim_type", "status", "claim_id"]).reset_index(drop=True)

display(claims_table_df)

,claim_id,claim_type,theme,status,rationale,evidence_tables
0,H1,hypothesis,readability_vs_faithfulness,Partial,Readability comparison is available with avera...,readability_summary.csv; compare_condition_sum...
1,H2,hypothesis,constraint_value,Partial,"Validator summaries are available, but unconst...",validator_summary.csv; validator_failure_summa...
2,H3,hypothesis,drift_messaging,Partial,"Drift-related summaries exist, but the current...",drift_summary.csv; thinfile_summary.csv
3,H4,hypothesis,thin_file_governance,Partial,"Thin-file analysis is available, but comparati...",thinfile_summary.csv
4,RQ1,research_question,faithfulness,Supported,Faithfulness table populated with overlap@5=1....,faithfulness_summary.csv
5,RQ2,research_question,stability,Partial,Perturbation summary is available only at the ...,perturbation_response_summary.csv; perturbatio...
6,RQ3,research_question,decision_utility,Partial,No human simulatability study is present in th...,faithfulness_summary.csv; thinfile_summary.csv...
7,RQ4,research_question,drift_awareness,Partial,"Drift summary is populated from EO.monitoring,...",drift_summary.csv
8,RQ5,research_question,thin_file_robustness,Partial,"Thin-file summary is available, but the compar...",thinfile_summary.csv
9,RQ6,research_question,portability_optional,Not yet tested,Optional portability slices are not present in...,


In [364]:
# -------------------------------------------------
# Safe export block for thesis tables
# -------------------------------------------------

EXPORT_DIR = OUTPUT / "thesis_tables"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

def _safe_export_df(df_name: str, filename: str, columns=None):
    if df_name not in globals() or globals()[df_name] is None:
        globals()[df_name] = pd.DataFrame(columns=columns or [])
        print(f"[guard] {df_name} missing -> exporting empty dataframe")
    df = globals()[df_name]
    out_path = EXPORT_DIR / filename
    df.to_csv(out_path, index=False)
    print(f"[export] {df_name}: {len(df)} rows -> {out_path}")

_safe_export_df("rq_hypothesis_coverage_df", "rq_hypothesis_coverage.csv")
_safe_export_df("artifact_grounding_summary_df", "artifact_grounding_summary.csv")
_safe_export_df("thinfile_action_governance_df", "thinfile_action_governance.csv")
_safe_export_df("baseline_run_summary_df", "baseline_run_summary.csv")
_safe_export_df("baseline_metrics_df", "baseline_metrics.csv")
_safe_export_df("validator_summary_df", "validator_summary.csv")
_safe_export_df("validator_failure_summary_df", "validator_failure_summary.csv")
_safe_export_df("perturbation_response_summary_df", "perturbation_response_summary.csv")
_safe_export_df("perturbation_build_summary_df", "perturbation_build_summary.csv")
_safe_export_df("faithfulness_summary_df", "faithfulness_summary.csv")
_safe_export_df("stability_summary_df", "stability_summary.csv")
_safe_export_df("thinfile_summary_df", "thinfile_summary.csv")
_safe_export_df("thinfile_delta_summary_df", "thinfile_delta_summary.csv")
_safe_export_df("drift_summary_df", "drift_summary.csv")
_safe_export_df("readability_summary_df", "readability_summary.csv")
_safe_export_df("compare_condition_summary_df", "compare_condition_summary.csv")
_safe_export_df("claims_table_df", "claims_table.csv")

print(f"\n[done] Safe thesis table export completed: {EXPORT_DIR}")

[export] rq_hypothesis_coverage_df: 10 rows -> /workspaces/miniature-fishstick/output/thesis_tables/rq_hypothesis_coverage.csv
[export] artifact_grounding_summary_df: 3 rows -> /workspaces/miniature-fishstick/output/thesis_tables/artifact_grounding_summary.csv
[export] thinfile_action_governance_df: 3 rows -> /workspaces/miniature-fishstick/output/thesis_tables/thinfile_action_governance.csv
[export] baseline_run_summary_df: 1 rows -> /workspaces/miniature-fishstick/output/thesis_tables/baseline_run_summary.csv
[export] baseline_metrics_df: 2 rows -> /workspaces/miniature-fishstick/output/thesis_tables/baseline_metrics.csv
[export] validator_summary_df: 1 rows -> /workspaces/miniature-fishstick/output/thesis_tables/validator_summary.csv
[export] validator_failure_summary_df: 1 rows -> /workspaces/miniature-fishstick/output/thesis_tables/validator_failure_summary.csv
[export] perturbation_response_summary_df: 3 rows -> /workspaces/miniature-fishstick/output/thesis_tables/perturbation_re

In [365]:
# ============================================================
# Thesis-ready Results / Discussion narrative from claims table
# ============================================================

def _claim_status(claim_id):
    row = claims_table_df.loc[claims_table_df["claim_id"] == claim_id]
    if len(row):
        return str(row["status"].iloc[0])
    return "NA"

def _claim_rationale(claim_id):
    row = claims_table_df.loc[claims_table_df["claim_id"] == claim_id]
    if len(row):
        return str(row["rationale"].iloc[0])
    return ""

rq1_status = _claim_status("RQ1")
rq2_status = _claim_status("RQ2")
rq3_status = _claim_status("RQ3")
rq4_status = _claim_status("RQ4")
rq5_status = _claim_status("RQ5")
rq6_status = _claim_status("RQ6")

h1_status = _claim_status("H1")
h2_status = _claim_status("H2")
h3_status = _claim_status("H3")
h4_status = _claim_status("H4")

rq1_overlap = faithfulness_summary_df["mean_overlap_at_5"].iloc[0] if "mean_overlap_at_5" in faithfulness_summary_df.columns and not faithfulness_summary_df.empty else None
rq1_dir = faithfulness_summary_df["mean_direction_accuracy"].iloc[0] if "mean_direction_accuracy" in faithfulness_summary_df.columns and not faithfulness_summary_df.empty else None
rq1_action = faithfulness_summary_df["mean_action_consistency"].iloc[0] if "mean_action_consistency" in faithfulness_summary_df.columns and not faithfulness_summary_df.empty else None

print("## Thesis-ready results summary\n")

print(
    "The current notebook run provides strongest support for RQ1 (faithfulness). "
    f"On the EO-linked evaluation subset, constrained LLM narratives achieved overlap@5={rq1_overlap:.3f}, "
    f"direction accuracy={rq1_dir:.3f}, and action consistency={rq1_action:.3f}. "
    "Within the current evaluated sample, this indicates that the generated narratives preserved the ranked EO driver set, "
    "maintained driver direction, and matched the EO-recommended action class."
)
print()

print(
    "Evidence for RQ2 (stability) is currently partial. Perturbation artifacts are available only at the EO level, "
    "because perturbed LLM narrative outputs are absent in the present run. As a result, the notebook can support "
    "partial stability analysis of EO/action changes under perturbation, but not full narrative-response stability."
)
print()

print(
    "Evidence for RQ3 (decision utility) is also partial. No human simulatability study is implemented in the current notebook state, "
    "so utility is evidenced indirectly through the proxy suite: faithfulness metrics, disclosure behavior, and action consistency."
)
print()

print(
    "Evidence for RQ4 (drift awareness) is partial. Drift-stratified summaries are available from the nested EO monitoring fields, "
    "which supports governance-oriented reporting of WARN vs non-WARN conditions. However, the present notebook does not yet show a strong "
    "causal relationship between drift disclosure and safer or more calibrated downstream action behavior."
)
print()

print(
    "Evidence for RQ5 (thin-file robustness) is partial but substantively relevant. Thin-file summaries are available, "
    "but comparative thin-file versus non-thin-file deltas remain incomplete in the current evaluation slice. "
    "This limits the strength of any relative claim, although the notebook does support direct analysis of narrative behavior in sparse-evidence regimes."
)
print()

print(
    "RQ6 (portability) remains untested in the current notebook run. The optional cyber or crypto portability slices described in the proposal "
    "have not yet been executed."
)
print()

print(
    "At the hypothesis level, H1 is partially supported insofar as a basic readability comparison is available, "
    "but richer readability measures or human judgments would be needed for a strong conclusion. H2 is partial because validator summaries exist, "
    "but unconstrained ablation evidence is not yet clearly represented in the current result pack. H3 remains partial because drift-related reporting exists "
    "without a strong demonstrated effect on action calibration. H4 is partial because thin-file analysis is available, but comparative evidence remains incomplete."
)
print()

print(
    "Overall, the present notebook run already supports a credible minimum-viable thesis result: "
    "a stable automated evaluation pipeline with strong faithfulness evidence for EO-grounded constrained narratives, "
    "plus partial but useful governance-oriented evidence for thin-file handling, drift reporting, validator behavior, and perturbation sensitivity."
)

## Thesis-ready results summary

The current notebook run provides strongest support for RQ1 (faithfulness). On the EO-linked evaluation subset, constrained LLM narratives achieved overlap@5=1.000, direction accuracy=1.000, and action consistency=1.000. Within the current evaluated sample, this indicates that the generated narratives preserved the ranked EO driver set, maintained driver direction, and matched the EO-recommended action class.

Evidence for RQ2 (stability) is currently partial. Perturbation artifacts are available only at the EO level, because perturbed LLM narrative outputs are absent in the present run. As a result, the notebook can support partial stability analysis of EO/action changes under perturbation, but not full narrative-response stability.

Evidence for RQ3 (decision utility) is also partial. No human simulatability study is implemented in the current notebook state, so utility is evidenced indirectly through the proxy suite: faithfulness metrics, disclosure 

## Cross-condition comparison note

This comparison does not attempt to measure readability or human preference. Instead, it tests whether constrained LLM narratives remain structurally aligned with the deterministic template baseline on shared EO inputs.